# || NEMO Workstation ||
© Konstantinos Andreadis 2024 (PhD in the Roux Lab & Salbreux Lab at UNIGE, Switzerland)

please cite: K.Andreadis _et al._ "NEMO: Mesh-Based Tangential Nematic Field, Defect, and Morphology Analysis in Volumetric Microscopy Data" (2026, _in preparation_)

In [ ]:
# Import custom scripts
from importlib import reload

from scripts import analysis, datahandler, visuals, simulation

for module in (analysis, datahandler, visuals, simulation):
    reload(module)

# Import python essentials
import os
import numpy as np
import trimesh

In [ ]:
# Initialise Napari viewer once
visuals.view_mesh([])

# --Import Image--

In [ ]:
# ==== Choose Image ====
# [!] WINDOWS: Sometimes the r before the file path string is needed, no idea why.
img_path = '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/72/300/Gas2.tif'
print(f"Selected image path: {img_path}")

# ==== Choose Time Step and Channel ====
t_select = 0
c_select = 0

# ==== [Optional] Reduce Resolution ====
z_reduce_factor = 1  # 1 means no reduction
xy_reduce_factor = 1  # 1 means no reduction

# ==== [Optional] Normalise Intensities to [0, 1] ====
normalise_intensities = False  # can be set to False

# ==== [Optional] Overwrite Scaling with FIJI Values ====
custom_scaling = None  # please use (z, y, x)

# ==== [Optional] Overwrite unit with FIJI Values ====
custom_unit = "um"  # as string

# ==== Load Image ====
img_load = analysis.load_img_virtual(path=img_path, t_sel_idx=t_select, c_sel_idx=c_select, reduce_xy=xy_reduce_factor,
                                     reduce_z=z_reduce_factor, norm_vals=normalise_intensities,
                                     custom_scaling=custom_scaling, custom_unit=custom_unit)
if img_load is not None:
    img_raw, img_dim, img_scale, img_unit = img_load

    # ==== Create Folder Structure ====
    resdata_dir, resfig_dir = datahandler.create_resdirs(img_path)
    # resdata_dir, resfig_dir = datahandler.create_resdirs(img_path, ct_label=f"t={t_select}_c={c_select}")

    # ==== Plot Image Slices and Max Projections ====
    visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit, max_proj=True, cmap='Greens',
                     savefig=os.path.join(resfig_dir, "sliced_maxproj_raw.png"))
    # z_i, y_i, x_i = int(200 / img_scale[0]), int(570 / img_scale[1]), int(455 / img_scale[2])
    z_i, y_i, x_i = int(img_dim[0] // 2), int(img_dim[1] // 2), int(img_dim[2] // 2)
    visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit, x_i=x_i, y_i=y_i, z_i=z_i,
                     savefig=os.path.join(resfig_dir, "sliced_raw.png"), cmap="Greens_r")
else:
    # ==== Default scenario ====
    img_raw, img_dim, img_scale, img_unit = np.zeros((1, 1, 1)), (1, 1, 1), (1, 1, 1), "?"
    resdata_dir, resfig_dir = None, None

In [ ]:
visuals.view_colored_labels_3d(segmentation_3d=img_raw.astype(int), scale=img_scale)

In [ ]:
# ==== 3D Render Image ====
visuals.view_img(img_list=[img_raw], title_list=["Raw Image"], scale=img_scale)

## Channel Composite Viewer

In [ ]:
# ==== Choose Time Point ====
t_select = 0
dims = analysis.load_img_dimensions(img_path)
num_channels = dims["C"]
channel_colors = ["Greens", "Reds", "Blues"]
channel_labels = ["Channel 1", "Channel 2", "Channel 3"]
composite_stack = []

for c_i in range(num_channels):
    # ==== Choose Time Step and Channel ====
    c_select = c_i

    # ==== Load Image ====
    img_load = analysis.load_img_virtual(path=img_path, t_sel_idx=t_select,
                                         c_sel_idx=c_select, reduce_xy=1, reduce_z=1, norm_vals=False,
                                         custom_scaling=None, custom_unit="um")
    img_raw_i, img_dim_i, img_scale_i, img_unit_i = img_load
    composite_stack.append(img_raw_i)

    # # ==== Plot Image Slices and Max Projections ====
    visuals.plot_img(img=img_raw_i, scale=img_scale_i, unit=img_unit_i, max_proj=True, cmap=f"{channel_colors[c_i]}",
                     savefig=os.path.join(resfig_dir, f"c={c_i}_sliced_maxproj_raw.png"))
    z_i, y_i, x_i = int(img_dim[0] // 2), int(img_dim[1] // 2), int(img_dim[2] // 2)
    visuals.plot_img(img=img_raw_i, scale=img_scale_i, unit=img_unit_i, x_i=x_i, y_i=y_i, z_i=z_i,
                     cmap=f"{channel_colors[c_i]}_r", savefig=os.path.join(resfig_dir, f"c={c_i}_sliced_raw.png"))
# ==== 3D Render Image ====
visuals.view_img(img_list=composite_stack, title_list=channel_labels, color_list=[f"{i}_r" for i in channel_colors],
                 scale=img_scale, opacity_list=[0.8 for i in channel_colors])

## Live Channel Viewer

In [ ]:
# ==== Choose Channel ====
dims = analysis.load_img_dimensions(img_path)
num_timepoints = dims["T"]
num_channels = dims["C"]

video_stack = []
for c_i in range(num_channels):
    time_stack = []
    for t_i in range(num_timepoints):
        # ==== Load Image ====
        img_load = analysis.load_img_virtual(path=img_path, t_sel_idx=t_i,
                                             c_sel_idx=c_i, reduce_xy=1, reduce_z=1, norm_vals=False,
                                             custom_scaling=None, custom_unit="um")
        img_raw_i, img_dim_i, img_scale_i, img_unit_i = img_load
        time_stack.append(img_raw_i)
    time_stack = np.stack(time_stack, axis=0)
    video_stack.append(time_stack)

# ==== 3D Render Image ====
visuals.view_img(img_list=video_stack, scale=img_scale)

In [ ]:
# ==== Choose Channel ====
c_select = 0
dims = analysis.load_img_dimensions(img_path)
num_timepoints = dims["T"]
time_stack = []

for t_i in range(num_timepoints):
    # ==== Load Image ====
    img_load = analysis.load_img_virtual(path=img_path, t_sel_idx=t_i,
                                         c_sel_idx=c_select, reduce_xy=1, reduce_z=1, norm_vals=False,
                                         custom_scaling=None, custom_unit="um")
    img_raw_i, img_dim_i, img_scale_i, img_unit_i = img_load
    time_stack.append(img_raw_i)
time_stack = np.stack(time_stack, axis=0)
# ==== 3D Render Image ====
visuals.view_img(img_list=[time_stack], scale=img_scale)

# -- Create Mesh --

## |1| Image Blur & Threshold

In [ ]:
# ==== Blur Image ====
sigma = 6
sigma_ = analysis.rescale_val_xyz(val=sigma, scale=img_scale)
img_blur = analysis.gaussian_blur(img=img_raw, sigma=sigma_, renorm=False)

# ==== Plot Image Slices ====
visuals.plot_img(img=img_blur, scale=img_scale, unit=img_unit, cmap="inferno",
                 savefig=os.path.join(resfig_dir, "sliced_blur.png"))

# ==== 3D Render Image ====
# visuals.view_img([img_blur], scale=img_scale, title_list=["Blurred Image"])

In [ ]:
# ==== Yen Threshold Image ====
# img_thresh_val = np.min([analysis.yen_thresh(img_blur[:, :, img_dim[2] // i]) for i in np.arange(2, 6)])
img_thresh_val = np.min(
    [analysis.yen_thresh(img_blur[:, :, img_dim[2] // 2]),
     analysis.yen_thresh(img_blur[:, img_dim[1] // 2, :]),
     analysis.yen_thresh(img_blur[img_dim[0] // 2, :, :])]) * 1

# ==== Binarise Image using Threshold ====
img_thresh = analysis.thresh_img(img_blur, img_thresh_val)

# ==== Plot Image Slices ====
visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit, savefig=os.path.join(resfig_dir, "sliced_thresh.png"),
                 cmap="inferno", thresh_mask=img_thresh)

In [ ]:
# ==== 3D Render Image ====
visuals.view_img([img_raw, img_thresh], scale=img_scale, title_list=["Raw Image", "Thresholded Image"],
                 color_list=["Greens_r", "Blues_r"], opacity_list=[1.0, 0.7])

In [ ]:
# # ==== [Optional] Fill Holes in Binary Image ====
# img_thresh = analysis.fill_holes_img(img_thresh)
#
# # ==== Plot Image Slices ====
# visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit,
#                  savefig=os.path.join(resfig_dir, "sliced_thresh.png"),
#                  cmap="inferno", thresh_mask=img_thresh)

In [ ]:
# # ==== [Optional] Save Thresholded Raw Image as .tiff ====
# visuals.view_img([img_raw, img_thresh], opacity_list=[1.0, 0.6], color_list=["green", "Blues"], scale=img_scale,
#                  title_list=["Raw Image", "Thresholded Image"])
# img_thresh = analysis.thresh_img(img_raw, img_thresh_val, keep_values_above=True)
# datahandler.save_tiff(img_thresh, filepath=os.path.join(resfig_dir, "thresh_img.tiff"))

## |2| Surface Mesh Extraction

In [ ]:
# ==== Segment Surface Mesh(es) ====
mcub_res = 2
full_mesh = analysis.marching_cubes(img=img_thresh, scale=img_scale, level=0.5, step_size=mcub_res)

# ==== Select INNER / OUTER Mesh ====
# mesh_sel_mask = np.einsum('ij,ij->i', full_mesh.vertices - np.mean(full_mesh.vertices, axis=0),
#                           full_mesh.vertex_normals) > 0

# ==== Select TOP / BOTTOM Mesh ====
mesh_sel_mask = np.einsum('ij,ij->i', np.array([[1, 0, 0] for _ in range(len(full_mesh.vertices))]),
                          full_mesh.vertex_normals) < 0

# ==== Apply Sub-Mesh Selection ====
inner_mesh = analysis.sel_submesh(mesh=full_mesh, mask=mesh_sel_mask)
outer_mesh = analysis.sel_submesh(mesh=full_mesh, mask=~mesh_sel_mask)
# inner_mesh = analysis.find_connected_meshes(mesh=full_mesh)[0]

# ==== Save Mesh(es) ====
datahandler.save_mesh(full_mesh, os.path.join(resdata_dir, "full_mesh.ply"))
datahandler.save_mesh(inner_mesh, os.path.join(resdata_dir, "inner_mesh.ply"))
datahandler.save_mesh(outer_mesh, os.path.join(resdata_dir, "outer_mesh.ply"))
print(
    f"Number of vertices: #INNER = {inner_mesh.vertices.shape[0]} + #OUTER = {outer_mesh.vertices.shape[0]} == #FULL = {full_mesh.vertices.shape[0]}!")

# ==== Plot Image Slices with Mesh Overlay ====
visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit,
                 savefig=os.path.join(resfig_dir, "sliced_raw_full-mesh.png"), meshes=[inner_mesh, outer_mesh],
                 mesh_colors=["red", "blue"])

In [ ]:
# ==== 3D Render Mesh(es) ====
# visuals.view_mesh(mesh_list=[outer_mesh, inner_mesh], mesh_colors=["white", "red"],
#                   mesh_titles=["Outer Mesh", "Inner Mesh"], mesh_opacities=[0.3, 0.3],
#                   img=img_raw, img_opacity=0.5, scale=img_scale)
visuals.view_mesh(mesh_list=[full_mesh], mesh_colors=["white"],
                  mesh_titles=["Full Mesh"], mesh_opacities=[1.0], vec_freq=5,
                  img=img_raw, img_opacity=0.9, scale=img_scale)

## |3| Mesh Processing

In [ ]:
# ==== Smooth Mesh(es) ====
smooth_factor = 0.001
smooth_iterations = 100
full_mesh_smooth = analysis.taubin_smooth_mesh(mesh=full_mesh, n_iter=smooth_iterations, pass_band=smooth_factor)
inner_mesh_smooth = analysis.taubin_smooth_mesh(mesh=inner_mesh, n_iter=smooth_iterations, pass_band=smooth_factor)
outer_mesh_smooth = analysis.taubin_smooth_mesh(mesh=outer_mesh, n_iter=smooth_iterations, pass_band=smooth_factor)

# ==== Sub-Sample Mesh(es) ====
# full_mesh_smooth = analysis.subdivide_mesh(mesh=full_mesh_smooth, max_edge=2)
# inner_mesh_smooth = analysis.subdivide_mesh(mesh=inner_mesh_smooth, max_edge=2)
# outer_mesh_smooth = analysis.subdivide_mesh(mesh=outer_mesh_smooth, max_edge=2)

# ==== Save Mesh(es) ====
datahandler.save_mesh(full_mesh_smooth, os.path.join(resdata_dir, "full_mesh_smooth.ply"))
datahandler.save_mesh(inner_mesh_smooth, os.path.join(resdata_dir, "inner_mesh_smooth.ply"))
datahandler.save_mesh(outer_mesh_smooth, os.path.join(resdata_dir, "outer_mesh_smooth.ply"))

# ==== Plot Image Slices with Mesh Overlay ====
visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit,
                 savefig=os.path.join(resfig_dir, "sliced_smooth_full-mesh.png"), meshes=[full_mesh, full_mesh_smooth],
                 mesh_colors=["black", "purple"])
visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit,
                 savefig=os.path.join(resfig_dir, "sliced_smooth_inner-mesh.png"),
                 meshes=[inner_mesh, inner_mesh_smooth], mesh_colors=["black", "red"])
visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit,
                 savefig=os.path.join(resfig_dir, "sliced_smooth_outer-mesh.png"),
                 meshes=[outer_mesh, outer_mesh_smooth], mesh_colors=["black", "blue"])

In [ ]:
# ==== 3D Render Mesh(es) ====
visuals.view_mesh(mesh_list=[inner_mesh_smooth, outer_mesh_smooth], mesh_colors=["red", "white"],
                  mesh_titles=["Smooth Inner Mesh", "Smooth Outer Mesh"], mesh_opacities=[0.3, 0.3],
                  img=img_raw, img_opacity=0.5, scale=img_scale)

# visuals.view_mesh(mesh_list=[inner_mesh, inner_mesh_smooth], mesh_colors=["grey", "red"],
#                   mesh_titles=["Inner Mesh", "Smooth Inner Mesh"], mesh_opacities=[0.3, 0.3])
# visuals.view_mesh(mesh_list=[full_mesh, full_mesh_smooth], mesh_colors=["grey", "red"],
#                   mesh_titles=["Full Mesh", "Smooth Full Mesh"], mesh_opacities=[0.3, 0.3], img=img_raw,
#                   scale=img_scale)

### |3.1| Select Largest Mesh

In [ ]:
full_mesh_smooth_subset_all = analysis.find_connected_meshes(mesh=full_mesh_smooth)
sizes = [mesh.vertices.shape[0] for mesh in full_mesh_smooth_subset_all]
full_mesh_smooth_subset = full_mesh_smooth_subset_all[np.argmax(sizes)]
visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit,
                 savefig=os.path.join(resfig_dir, "sliced_smooth_full-mesh_sub.png"), meshes=[full_mesh_smooth_subset],
                 mesh_colors=["purple"])

inner_mesh_smooth_subset_all = analysis.find_connected_meshes(mesh=inner_mesh_smooth)
sizes = [mesh.vertices.shape[0] for mesh in inner_mesh_smooth_subset_all]
inner_mesh_smooth_subset = inner_mesh_smooth_subset_all[np.argmax(sizes)]
visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit,
                 savefig=os.path.join(resfig_dir, "sliced_smooth_inner-mesh_sub.png"),
                 meshes=[inner_mesh_smooth_subset],
                 mesh_colors=["red"])

outer_mesh_smooth_subset_all = analysis.find_connected_meshes(mesh=outer_mesh_smooth)
sizes = [mesh.vertices.shape[0] for mesh in outer_mesh_smooth_subset_all]
outer_mesh_smooth_subset = outer_mesh_smooth_subset_all[np.argmax(sizes)]
visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit,
                 savefig=os.path.join(resfig_dir, "sliced_smooth_outer-mesh_sub.png"),
                 meshes=[outer_mesh_smooth_subset],
                 mesh_colors=["blue"])

In [ ]:
visuals.view_mesh(mesh_list=[full_mesh_smooth, full_mesh_smooth_subset], mesh_colors=["white", "red"],
                  mesh_titles=["Smooth Full Mesh", "Largest Subset of Full Mesh"], mesh_opacities=[0.8, 1.0],
                  vec_freq=5,
                  img=img_raw, img_opacity=0.5, scale=img_scale)

### |3.2| EMBL Sphere Fit

In [ ]:
mesh_to_fit = full_mesh_smooth.copy()
# ==== Fit Sphere ====
sphere_params = analysis.fit_sphere(points=mesh_to_fit.vertices)
sphere_x0, sphere_y0, sphere_z0, sphere_radius = sphere_params
sphere_mesh = trimesh.creation.icosphere(radius=sphere_radius, subdivisions=8)
sphere_mesh.vertices += [sphere_x0, sphere_y0, sphere_z0]
datahandler.save_array(np.array(sphere_params)[:, np.newaxis].T, "sphere_fit", header="x,y,z,radius",
                       folderpath=resdata_dir)
# ==== Crop (Part of) Sphere ====
seg_fit_crop_cap_angle = 120.0
print(f">> Cropping sphere to {seg_fit_crop_cap_angle} degrees cap...")
sphere_crop_mask = ((sphere_mesh.vertices[:, 0] - sphere_x0) / np.linalg.norm(
    sphere_mesh.vertices - [sphere_x0, sphere_y0, sphere_z0], axis=1)) >= np.cos(
    np.radians(seg_fit_crop_cap_angle))
sphere_mesh_cropped = analysis.sel_submesh(mesh=sphere_mesh, mask=sphere_crop_mask)
print(f"Num of sphere vertices: {sphere_mesh_cropped.vertices.shape[0]}")

# ==== Plot Image Slices with Mesh Overlay ====
visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit,
                 savefig=os.path.join(resfig_dir, "sliced_raw_sphere-fit.png"), cmap="Greens_r", show_mesh_normals=True,
                 meshes=[sphere_mesh_cropped], mesh_alpha=1.0)

In [ ]:
# ==== 3D Render Mesh(es) ====
visuals.view_mesh([sphere_mesh, sphere_mesh_cropped], mesh_opacities=[0.5, 1.0], mesh_colors=["white", "red"],
                  img=img_raw, scale=img_scale)

## |5| Save Sampling Mesh

In [ ]:
# ==== Select Sampling Mesh ====
sampl_mesh = full_mesh_smooth_subset.copy()  # full_mesh_smooth, outer_mesh_smooth, inner_mesh_smooth

sampl_mesh_name = "sampling_mesh.ply"
# ==== [Optional] Merge Meshes Instead ====
# mesh_merge_1 = datahandler.load_mesh(os.path.join(resdata_dir, "inner_mesh_smooth.ply"), recalc_normals=False)
# mesh_merge_2 = datahandler.load_mesh(os.path.join(resdata_dir, "outer_mesh_smooth.ply"), recalc_normals=False)
# mesh_merge_1.invert()
# sampl_mesh = trimesh.util.concatenate([mesh_merge_1, mesh_merge_2])

# ==== Save Sampling Mesh ====
datahandler.save_mesh(sampl_mesh, os.path.join(resdata_dir, sampl_mesh_name))

# ==== Plot Image Slices with Mesh Overlay ====
visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit, meshes=[sampl_mesh],
                 slice_depth=1, show_mesh_normals=True,
                 savefig=os.path.join(resfig_dir, "sliced_raw_sampling-mesh.png"))
mesh_slice_max = np.array([img_scale[i] * img_dim[i] for i in range(len(img_scale))]).max() / 2
visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit, meshes=[sampl_mesh],
                 slice_depth=mesh_slice_max, mesh_alpha=0.02, max_proj=True,
                 savefig=os.path.join(resfig_dir, "sliced"
                                                  "_raw_sampling-mesh_maxproj.png"))
print(f"Number of sampling points: {len(sampl_mesh.vertices)} !")

# ==== 3D Render Mesh(es) ====
# visuals.view_mesh(mesh_list=[sampl_mesh], mesh_titles=["Mesh"], mesh_colors=["white"], mesh_opacities=[0.4],
#                   img=img_raw, scale=img_scale, vec_freq=100, hide_vectors=False, vec_length=1)

# -- Load Mesh --

In [ ]:
# ==== Load Sampling Mesh ====
sampl_mesh_name = "sampling_mesh.ply"
# sampl_mesh_name = "outer_mesh.ply"
mesh_path = os.path.join(resdata_dir, sampl_mesh_name)
sampl_mesh = datahandler.load_mesh(mesh_path, recalc_normals=True, clean=False)
print(f"Number of sampling vertices: {len(sampl_mesh.vertices)} !")
sampl_mesh.invert()

# ==== Plot Image Slices with Mesh Overlay ====
visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit, meshes=[sampl_mesh],
                 slice_depth=1, show_mesh_normals=True,
                 savefig=os.path.join(resfig_dir, f"sliced_raw_{sampl_mesh_name.split('.ply')[0]}.png"))
mesh_slice_max = np.array([img_scale[i] * img_dim[i] for i in range(len(img_scale))]).max() / 2
visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit, meshes=[sampl_mesh],
                 slice_depth=mesh_slice_max, mesh_alpha=0.02, max_proj=True,
                 savefig=os.path.join(resfig_dir, f"sliced_raw_{sampl_mesh_name.split('.ply')[0]}-mesh_maxproj.png"))
print(f"Number of sampling points: {len(sampl_mesh.vertices)} !")

# ==== 3D Render Mesh(es) ====
# visuals.view_mesh(mesh_list=[sampl_mesh], mesh_titles=["Mesh"], mesh_colors=["white"], mesh_opacities=[0.4],
#                   img=img_raw, scale=img_scale, vec_freq=400, hide_vectors=True, vec_length=20)
# visuals.view_mesh(mesh_list=[sampl_mesh], mesh_titles=["Mesh"], mesh_colors=["white"], mesh_opacities=[1.0],
#                   img=img_raw, scale=img_scale, vec_freq=100, hide_vectors=False, vec_length=5, vec_edge_width=0.1)

In [ ]:
visuals.view_mesh(mesh_list=[sampl_mesh], mesh_titles=["Mesh"], mesh_colors=["white"], mesh_opacities=[1.0],
                  img=img_raw, scale=img_scale, vec_freq=100, hide_vectors=False, vec_length=10, vec_edge_width=0.2)

In [ ]:
name_mesh_1 = "inner_mesh.ply"
name_mesh_2 = "outer_mesh.ply"
mesh_1 = datahandler.load_mesh(os.path.join(resdata_dir, name_mesh_1), recalc_normals=True, clean=False)
mesh_2 = datahandler.load_mesh(os.path.join(resdata_dir, name_mesh_2), recalc_normals=True, clean=False)
mesh_2.invert()
# visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit, meshes=[mesh_1, mesh_2],
#                  mesh_colors=["red", "blue"], slice_depth=1, show_mesh_normals=True, normal_scale=0.1,
#                  normal_vecfreq=30, mesh_thick=0.05,
#                  savefig=os.path.join(resfig_dir, "sliced_raw_inner-outer-mesh.png"))

In [ ]:
visuals.view_mesh(mesh_list=[mesh_1, mesh_2],
                  mesh_colors=["Red", "Blue"],
                  mesh_titles=["MESH 1", "MESH 2"], hide_vectors=False, vec_length=7,
                  vec_freq=100, img=img_raw, img_opacity=0.5, scale=img_scale)

# -- Mesh Analysis --

## |1| Dual Mesh Distance/Thickness

In [ ]:
# img_unit = "px"
# resfig_dir = '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!nemo/control_meshes'
# resdata_dir = '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!nemo/control_meshes'
# mesh_1 = trimesh.creation.icosphere(radius=100.0, subdivisions=8)
# print(len(mesh_1.vertices))
# mesh_2 = trimesh.creation.icosphere(radius=130.0, subdivisions=8)
# mesh_2.invert()
# visuals.plot_img(img=np.zeros(shape=(1, 1, 1)), scale=(1, 1, 1), unit="px", meshes=[mesh_1, mesh_2], normal_vecfreq=20,
#                  mesh_colors=["red", "blue"], slice_depth=1, show_mesh_normals=True, normal_scale=0.05)

In [ ]:
# ==== Select Mesh(es) ====
name_mesh_1 = "inner_mesh.ply"
name_mesh_2 = "outer_mesh.ply"
mesh_1 = datahandler.load_mesh(os.path.join(resdata_dir, name_mesh_1), recalc_normals=True, clean=False)
mesh_2 = datahandler.load_mesh(os.path.join(resdata_dir, name_mesh_2), recalc_normals=True, clean=False)
mesh_2.invert()

datahandler.save_mesh(mesh_1, os.path.join(resdata_dir, "mesh_1_morph.ply"))
datahandler.save_mesh(mesh_2, os.path.join(resdata_dir, "mesh_2_morph.ply"))

# ==== Plot Image Slices with Mesh Overlay ====
visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit, meshes=[mesh_1, mesh_2],
                 mesh_colors=["red", "blue"], slice_depth=1, show_mesh_normals=True, normal_scale=0.05)

# ==== 3D Render Mesh(es) ====
# visuals.view_mesh(mesh_list=[mesh_1, mesh_2],
#                   mesh_colors=["Green", "Red"],
#                   mesh_titles=["MESH 1", "MESH 2"], img=img_raw, img_opacity=0.5, scale=img_scale)

In [ ]:
mesh_1_radii = np.linalg.norm(mesh_1.vertices - np.mean(mesh_1.vertices, axis=0), axis=-1)
mesh_1_radii_avg = np.mean(mesh_1_radii)

mesh_2_radii = np.linalg.norm(mesh_2.vertices - np.mean(mesh_2.vertices, axis=0), axis=-1)
mesh_2_radii_avg = np.mean(mesh_2_radii)
print(f"Approximate radius of inner mesh: {mesh_1_radii_avg} and of outer mesh: {mesh_2_radii_avg}")

In [ ]:
# mesh_2_cropped = analysis.sel_submesh(mesh_2, ~np.all(mesh_2.vertices > 0, axis=1))
visuals.view_mesh(mesh_list=[mesh_1, mesh_2],
                  mesh_colors=["Red", "Blue"],
                  mesh_titles=["MESH 1", "MESH 2"], hide_vectors=False, vec_length=7,
                  vec_freq=100)  #, img=img_raw, img_opacity=0.5, scale=img_scale)

In [ ]:
# ==== Calculate Inter-Mesh Distance ====
# thickness_crop_range = [0, 30]
thickness_crop_range = None
thickness_sampl_number = 5000
dist_vals, dist_idxs = analysis.inter_dist_mesh(mesh_1=mesh_1, mesh_2=mesh_2, num_sample=thickness_sampl_number,
                                                crop_range=thickness_crop_range, debug=True, allow_multiple_hits=False)
full_dist_vals = analysis.interpolate_on_mesh(mesh_1, dist_idxs, dist_vals, k=10)
# ==== Save Inter-Mesh Distance ====
datahandler.save_array(full_dist_vals, "thickness", header=f"dist ({img_unit})", folderpath=resdata_dir)

# ==== Plot Inter-Mesh Distance ====
visuals.plot_hist(array=full_dist_vals, title=f"Thickness AVG = {full_dist_vals.mean():.2e} {img_unit}",
                  xlim=thickness_crop_range, savefig=os.path.join(resfig_dir, "thickness_hist.png"))
visuals.plot_maxproj_pts(verts=mesh_1.vertices, unit=img_unit, colors=full_dist_vals, cmap="coolwarm",
                         hexsize=200, cmap_label=f"Thickness ({img_unit})",
                         savefig=os.path.join(resfig_dir, "thickness.png"), figsize=(15, 7))

In [ ]:
import matplotlib.ticker as ticker
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Setup the figure
fig, ax = plt.subplots(figsize=(7, 6))

# 2. Plot the Thickness Distribution (Violin only, no strip plot)
# full_dist_vals is your calculated thickness array
sns.violinplot(y=full_dist_vals, ax=ax, color='lightblue', inner="quartile", cut=0)

# 3. Add the Expected Value Line (30)
expected_thickness = mesh_2_radii_avg - mesh_1_radii_avg
ax.axhline(expected_thickness, color='red', linestyle='--', linewidth=2,
           label=f'Expected: {expected_thickness}.0')

# 4. Professional Formatting
ax.set_title("Inter-Mesh Thickness Distribution", fontsize=14)
ax.set_ylabel(f"Thickness ({img_unit})", fontsize=12)

# Ensure the Y-axis shows the micro-variations around 30
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.8f'))

# Optional: Tighten the Y-limit if the variance is extremely small
# ax.set_ylim(29.99999, 30.00001)

ax.legend(loc='upper right')
plt.tight_layout()

# 5. Save and Show
plt.savefig(os.path.join(resfig_dir, "thickness_violin_precision.png"), bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# ==== 3D Render Result ====
visuals.view_colored_mesh_multiple([mesh_1, mesh_2],
                                   [visuals.color_scalar(full_dist_vals, normalise=True, cmap="coolwarm"),
                                    "white"], mesh_blending_list=["opaque", "translucent"],
                                   mesh_opacity_list=[1.0, 0.3], img=img_raw, scale=img_scale)

## |2| Dual Mesh Gaussian & Mean Curvature

In [ ]:
# ==== Define Crop Range of Gauss & Mean Curvature ====
gauss_exp = 1 / (np.ptp(mesh_1.vertices, axis=0).mean() / 2) ** 2
mean_exp = 1 / (np.ptp(mesh_1.vertices, axis=0).mean() / 2)
print(f"Gauss should be around {gauss_exp:.2e} (1/{img_unit}^2), "
      f"Mean should be around {mean_exp:.2e} (1/{img_unit})")
gauss_order_min, gauss_order_max = round(np.log10(gauss_exp)) - 1, round(np.log10(gauss_exp)) + 1
mean_order_min, mean_order_max = round(np.log10(mean_exp)) - 1, round(np.log10(mean_exp)) + 1
gauss_crop_range = [0.01 * 10 ** gauss_order_min, 10 ** gauss_order_max]
mean_crop_range = [0.01 * 10 ** mean_order_min, 10 ** mean_order_max]

gauss_crop_range, mean_crop_range = None, None

# ==== Define Number of Random Calculation Selection ====
curvature_num_samples = 5000

# ==== Define Number of Nearest Neighbours to use for surface fit ====
curvature_patch_info = ["nearest", 20]
# curvature_patch_info = ["radius", 100]
# ==== [Optional] Exclude Boundary Vertices ====
curvature_filter_boundary = False
curvature_filter_boundary_factor = 0.0  # exclude strength between 0 (max) and 1 (no exclusion)

### |2.1| Curvature Mesh #1

In [ ]:
# ==== Calculate Gauss & Mean Curvature ====
curvature_results = analysis.curvature_by_srf_fit(mesh_1, num_sample=curvature_num_samples,
                                                  patch_mode=curvature_patch_info[0],
                                                  patch_size=curvature_patch_info[1],
                                                  debug=True, filter_boundary=curvature_filter_boundary,
                                                  boundary_excl_factor=curvature_filter_boundary_factor,
                                                  gauss_crop_range=gauss_crop_range, mean_crop_range=mean_crop_range)

C_gauss, C_mean, C_gauss_idxs, C_mean_idxs = curvature_results
full_C_gauss_1 = analysis.interpolate_on_mesh(mesh_1, C_gauss_idxs, C_gauss, k=10)
full_C_mean_1 = analysis.interpolate_on_mesh(mesh_1, C_mean_idxs, C_mean, k=10)

# ==== Save Gauss & Mean Curvature ====
datahandler.save_array(full_C_gauss_1, "gauss_curv_1", header=f"gauss (1/{img_unit}^2)", folderpath=resdata_dir)
datahandler.save_array(full_C_mean_1, "mean_curv_1", header=f"mean (1/{img_unit})", folderpath=resdata_dir)

# ==== Plot Gauss & Mean Curvature ====
visuals.plot_hist(full_C_gauss_1, title=f"Gauss AVG = {full_C_gauss_1.mean():.2e} $(1/{img_unit}^2)$",
                  savefig=os.path.join(resfig_dir, "gauss_hist_1.png"))
visuals.plot_hist(full_C_mean_1, title=f"Mean AVG = {full_C_mean_1.mean():.2e} $(1/{img_unit})$",
                  savefig=os.path.join(resfig_dir, "mean_hist_1.png"))
visuals.plot_maxproj_pts(verts=mesh_1.vertices, unit=img_unit, colors=full_C_gauss_1, cmap="coolwarm",
                         hexsize=100, cmap_label=f"Gaussian Curvature $(1/{img_unit}^2)$",
                         savefig=os.path.join(resfig_dir, "gauss_curv_1.png"))
visuals.plot_maxproj_pts(verts=mesh_1.vertices, unit=img_unit, colors=full_C_mean_1, cmap="Spectral",
                         hexsize=100, cmap_label=f"Mean Curvature $(1/{img_unit})$",
                         savefig=os.path.join(resfig_dir, "mean_curv_1.png"))

In [ ]:
# ==== 3D Render Result ====
visuals.view_colored_mesh_multiple([mesh_1, mesh_1],
                                   [visuals.color_scalar(full_C_gauss_1, normalise=True, cmap="coolwarm"),
                                    visuals.color_scalar(full_C_mean_1, normalise=True, cmap="Spectral")],
                                   name_list=["Gauss", "Mean"])

### |2.1| Curvature Mesh #2

In [ ]:
# ==== Calculate Gauss & Mean Curvature ====
curvature_results = analysis.curvature_by_srf_fit(mesh_2, num_sample=curvature_num_samples,
                                                  patch_mode=curvature_patch_info[0],
                                                  patch_size=curvature_patch_info[1],
                                                  debug=True, filter_boundary=curvature_filter_boundary,
                                                  boundary_excl_factor=curvature_filter_boundary_factor,
                                                  gauss_crop_range=np.array(
                                                      gauss_crop_range) if gauss_crop_range is not None else None,
                                                  mean_crop_range=np.flip(np.array(
                                                      mean_crop_range) * -1) if mean_crop_range is not None else None)

C_gauss, C_mean, C_gauss_idxs, C_mean_idxs = curvature_results
full_C_gauss_2 = analysis.interpolate_on_mesh(mesh_2, C_gauss_idxs, C_gauss, k=10)
full_C_mean_2 = analysis.interpolate_on_mesh(mesh_2, C_mean_idxs, C_mean, k=10)
# ==== Save Gauss & Mean Curvature ====
datahandler.save_array(full_C_gauss_2, "gauss_curv_2", header=f"gauss (1/{img_unit}^2)", folderpath=resdata_dir)
datahandler.save_array(full_C_mean_2, "mean_curv_2", header=f"mean (1/{img_unit})", folderpath=resdata_dir)

# ==== Plot Gauss & Mean Curvature ====
visuals.plot_hist(full_C_gauss_2, title=f"Gauss AVG = {full_C_gauss_2.mean():.2e} $(1/{img_unit}^2)$",
                  savefig=os.path.join(resfig_dir, "gauss_hist_2.png"))
visuals.plot_hist(full_C_mean_2, title=f"Mean AVG = {full_C_mean_2.mean():.2e} $(1/{img_unit})$",
                  savefig=os.path.join(resfig_dir, "mean_hist_2.png"))
visuals.plot_maxproj_pts(verts=mesh_2.vertices, unit=img_unit, colors=full_C_gauss_2, cmap="coolwarm",
                         hexsize=100, cmap_label=f"Gaussian Curvature $(1/{img_unit}^2)$",
                         savefig=os.path.join(resfig_dir, "gauss_curv_2.png"))
visuals.plot_maxproj_pts(verts=mesh_2.vertices, unit=img_unit, colors=full_C_mean_2, cmap="Spectral",
                         hexsize=100, cmap_label=f"Mean Curvature $(1/{img_unit})$",
                         savefig=os.path.join(resfig_dir, "mean_curv_2.png"))

In [ ]:
# ==== 3D Render Result ====
visuals.view_colored_mesh_multiple([mesh_2, mesh_2],
                                   [visuals.color_scalar(full_C_gauss_2, normalise=True, cmap="coolwarm"),
                                    visuals.color_scalar(full_C_mean_2, normalise=True, cmap="Spectral")],
                                   name_list=["Gauss", "Mean"])

In [ ]:

import matplotlib.ticker as ticker
import pandas as pd

# 1. Define Theoretical Targets
r1, r2 = mesh_1_radii_avg, mesh_2_radii_avg
expected_map = {
    ('Mean', 'Group 1'): 1 / r1,
    ('Mean', 'Group 2'): -1 / r2,
    ('Gaussian', 'Group 1'): 1 / (r1 ** 2),
    ('Gaussian', 'Group 2'): 1 / (r2 ** 2)
}

# 2. Prepare Data
df = pd.DataFrame({
    'Value': list(full_C_mean_1) + list(full_C_mean_2) +
             list(full_C_gauss_1) + list(full_C_gauss_2),
    'Group': ['Group 1'] * len(full_C_mean_1) + ['Group 2'] * len(full_C_mean_2) +
             ['Group 1'] * len(full_C_gauss_1) + ['Group 2'] * len(full_C_gauss_2),
    'Type': ['Mean'] * (len(full_C_mean_1) + len(full_C_mean_2)) +
            ['Gaussian'] * (len(full_C_gauss_1) + len(full_C_gauss_2))
})

# 3. Create 2x2 Subplot (Top: Mean, Bottom: Gaussian)
fig, axes = plt.subplots(2, 2, figsize=(12, 10), sharey=False)

types = ['Mean', 'Gaussian']
groups = ['Group 1', 'Group 2']

for r, dtype in enumerate(types):
    for c, group in enumerate(groups):
        ax = axes[r, c]

        # Filter data for this specific subplot
        subset = df[(df['Type'] == dtype) & (df['Group'] == group)]

        # Plot Distribution
        sns.violinplot(data=subset, y='Value', ax=ax, inner="quartile", palette="pastel")
        # sns.stripplot(data=subset, y='Value', ax=ax, color='black', alpha=0.2, jitter=True)

        # Add Dashed Expected Line
        target = expected_map[(dtype, group)]
        ax.axhline(target, color='red', linestyle='--', linewidth=2, label=f'Target: {target:.6f}')

        # Formatting
        ax.set_title(f'{dtype}: {group}', fontsize=14)
        ax.set_ylabel('Curvature Value')
        ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.8f'))
        ax.legend(loc='upper right')

plt.tight_layout()
output_path = os.path.join(resfig_dir, "mean-gauss_2x2_detailed.png")
plt.savefig(output_path, bbox_inches='tight', dpi=300)
plt.show()

### |2.1| Curvature Mesh Comparative View

In [ ]:

import matplotlib.cm as cm

# 1. Define your limits (using your variables)
cmap_lims_render_mean = [np.min([full_C_mean_1.min(), full_C_mean_2.min()]),
                         np.max([full_C_mean_1.max(), full_C_mean_2.max()])]
cmap_lims_render_gauss = [np.min([full_C_gauss_1.min(), full_C_gauss_2.min()]),
                          np.max([full_C_gauss_1.max(), full_C_gauss_2.max()])]

fig, ax = plt.subplots(figsize=(3, 6))
ax.set_visible(False)  # Hide the actual plot area

# 2. Create the Mean Colorbar
sm_mean = cm.ScalarMappable(cmap='viridis')  # Replace with your mean cmap
sm_mean.set_array([])
sm_mean.set_clim(cmap_lims_render_mean[0], cmap_lims_render_mean[1])

cbar_mean = fig.colorbar(sm_mean, ax=ax, location='left', pad=0.5)
cbar_mean.set_label('Mean Scale', fontsize=12)

# 3. Create the Gauss Colorbar
sm_gauss = cm.ScalarMappable(cmap='magma')  # Replace with your gauss cmap
sm_gauss.set_array([])
sm_gauss.set_clim(cmap_lims_render_gauss[0], cmap_lims_render_gauss[1])

cbar_gauss = fig.colorbar(sm_gauss, ax=ax, location='right', pad=0.5)
cbar_gauss.set_label('Gauss Scale', fontsize=12)
plt.savefig(os.path.join(resfig_dir, "joint-curv-colorbars.png"), bbox_inches='tight')
plt.show()

In [ ]:
# ==== 3D Render Result ====
cmap_lims_render_mean = [np.min([full_C_mean_1.min(), full_C_mean_2.min()]),
                         np.max([full_C_mean_1.max(), full_C_mean_2.max()])]
cmap_lims_render_gauss = [np.min([full_C_gauss_1.min(), full_C_gauss_2.min()]),
                          np.max([full_C_gauss_1.max(), full_C_gauss_2.max()])]

visuals.view_colored_mesh_multiple([mesh_1, mesh_2, mesh_1, mesh_2],
                                   [visuals.color_scalar(full_C_mean_1, cmap="Spectral",
                                                         manual_vminmax=cmap_lims_render_mean),
                                    visuals.color_scalar(full_C_mean_2, cmap="Spectral",
                                                         manual_vminmax=cmap_lims_render_mean),
                                    visuals.color_scalar(full_C_gauss_1, cmap="coolwarm",
                                                         manual_vminmax=cmap_lims_render_gauss),
                                    visuals.color_scalar(full_C_gauss_2, cmap="coolwarm",
                                                         manual_vminmax=cmap_lims_render_gauss)],
                                   name_list=["Mean Mesh 1", "Mean Mesh 2", "Gauss Mesh 1", "Gauss Mesh 2"])

## |3| Single Mesh Gaussian & Mean Curvatures

In [ ]:
mesh_curv_name = "sampling_mesh.ply"
mesh_curv = datahandler.load_mesh(os.path.join(resdata_dir, mesh_curv_name), recalc_normals=True, clean=False)
visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit, meshes=[mesh_curv], normal_vecfreq=5, normal_scale=0.05,
                 slice_depth=1, show_mesh_normals=True)

# ==== Define Crop Range of Gauss & Mean Curvature ====
gauss_exp = 1 / (np.ptp(mesh_curv.vertices, axis=0).mean() / 2) ** 2
mean_exp = 1 / (np.ptp(mesh_curv.vertices, axis=0).mean() / 2)
print(f"Gauss should be around {gauss_exp:.2e} (1/{img_unit}^2), "
      f"Mean should be around {mean_exp:.2e} (1/{img_unit})")
gauss_order_min, gauss_order_max = round(np.log10(gauss_exp)) - 1, round(np.log10(gauss_exp)) + 1
mean_order_min, mean_order_max = round(np.log10(mean_exp)) - 1, round(np.log10(mean_exp)) + 1
gauss_crop_range = [0.01 * 10 ** gauss_order_min, 10 ** gauss_order_max]
mean_crop_range = [0.01 * 10 ** mean_order_min, 10 ** mean_order_max]

# gauss_crop_range, mean_crop_range = None, None

# ==== Define Number of Random Calculation Selection ====
curvature_num_samples = 3000

# ==== Define Number of Nearest Neighbours to use for surface fit ====
curvature_patch_info = ["nearest", 20]
# curvature_patch_info = ["radius", 100]
# ==== [Optional] Exclude Boundary Vertices ====
curvature_filter_boundary = False
curvature_filter_boundary_factor = 0.0  # exclude strength between 0 (max) and 1 (no exclusion)

# ==== Calculate Gauss & Mean Curvature ====
curvature_results = analysis.curvature_by_srf_fit(mesh_curv, num_sample=curvature_num_samples,
                                                  patch_mode=curvature_patch_info[0],
                                                  patch_size=curvature_patch_info[1],
                                                  debug=True, filter_boundary=curvature_filter_boundary,
                                                  boundary_excl_factor=curvature_filter_boundary_factor,
                                                  gauss_crop_range=gauss_crop_range, mean_crop_range=mean_crop_range)

C_gauss, C_mean, C_gauss_idxs, C_mean_idxs = curvature_results
full_C_gauss_1 = analysis.interpolate_on_mesh(mesh_curv, C_gauss_idxs, C_gauss, k=10)
full_C_mean_1 = analysis.interpolate_on_mesh(mesh_curv, C_mean_idxs, C_mean, k=10)

# ==== Save Gauss & Mean Curvature ====
datahandler.save_array(full_C_gauss_1, f"{mesh_curv_name}_gauss_curv", header=f"gauss (1/{img_unit}^2)",
                       folderpath=resdata_dir)
datahandler.save_array(full_C_mean_1, f"{mesh_curv_name}_mean_curv", header=f"mean (1/{img_unit})",
                       folderpath=resdata_dir)

# ==== Plot Gauss & Mean Curvature ====
visuals.plot_hist(full_C_gauss_1, title=f"Gauss AVG = {full_C_gauss_1.mean():.2e} $(1/{img_unit}^2)$",
                  savefig=os.path.join(resfig_dir, f"{mesh_curv_name}_gauss_hist.png"))
visuals.plot_hist(full_C_mean_1, title=f"Mean AVG = {full_C_mean_1.mean():.2e} $(1/{img_unit})$",
                  savefig=os.path.join(resfig_dir, f"{mesh_curv_name}_mean_hist.png"))
visuals.plot_maxproj_pts(verts=mesh_curv.vertices, unit=img_unit, colors=full_C_gauss_1, cmap="coolwarm",
                         hexsize=100, cmap_label=f"Gaussian Curvature $(1/{img_unit}^2)$",
                         savefig=os.path.join(resfig_dir, f"{mesh_curv_name}_gauss_curv.png"))
visuals.plot_maxproj_pts(verts=mesh_curv.vertices, unit=img_unit, colors=full_C_mean_1, cmap="Spectral",
                         hexsize=100, cmap_label=f"Mean Curvature $(1/{img_unit})$",
                         savefig=os.path.join(resfig_dir, f"{mesh_curv_name}_mean_curv.png"))

# Projection onto Mesh

## |1| Global Projection

In [ ]:
# ==== Define Projection Range ====
dist_min = 0
dist_max = 30
dist_num = int(abs(dist_max - dist_min) / np.round(np.min(img_scale) * 2, 2))
# dist_num = 50
proj_mode = "mean"  # "max" or "mean"

# ==== Specify Custom Minimum ====
dist_min_custom = None
# dist_min_custom = np.linspace(0, 5, sampl_mesh.vertices.shape[0])

# ==== Project onto Mesh ====
proj_broad = analysis.proj2mesh(img=img_raw, mesh=sampl_mesh, unit=img_unit, min_dist_per_vert=dist_min_custom,
                                scale=img_scale, min_dist=dist_min, max_dist=dist_max, num_dist=dist_num,
                                mode=proj_mode, show_proj=True,
                                savefig=os.path.join(resfig_dir, "distgraph_broad-scan.png"),
                                normalise=False)

# ==== Plot Projected Result ====
# visuals.plot_maxproj_pts(verts=sampl_mesh.vertices, colors=proj_broad, cmap="Greens", hexsize=200,
#                          savefig=os.path.join(resfig_dir, "maxproj_broad-scan.png"), unit=img_unit, figsize=(18, 8))

# ==== Plot Histogram of Projected Values ====
visuals.plot_hist(proj_broad, title="Intensities (a.u.)")

In [ ]:
# ==== 3D Render Projected Result on Scaled Mesh ====
# sampl_mesh_layer = analysis.scale_mesh(mesh=sampl_mesh, distance=dist_max)
sampl_mesh_layer = sampl_mesh.copy()
visuals.view_colored_mesh_multiple(mesh_list=[sampl_mesh_layer],
                                   vert_colors_list=[
                                       visuals.color_scalar(proj_broad / proj_broad.max(), cmap="inferno")],
                                   mesh_blending_list=["opaque"])  #, img=img_raw, scale=img_scale)

# ==== 3D Render Projected Result ====
# visuals.view_colored_mesh(mesh=sampl_mesh,
#                           vert_colors=visuals.color_scalar(proj_broad / proj_broad.max(), cmap="inferno"),  #Greens_r
#                           mesh_blending="opaque")  #, img=img_raw, scale=img_scale)

In [ ]:
# ==== Plot Spherical Projection ====
sph_proj_phi, sph_proj_theta = analysis.spherical_project(
    pts=sampl_mesh.vertices)  #, ref_point=[sphere_x0, sphere_y0, sphere_z0],rotate=[90, 0, 90])
visuals.plot_spherical_projection(phi=sph_proj_phi, theta=sph_proj_theta, intensities=proj_broad, ptview=False,
                                  hexgridsize=200, cmap="inferno",
                                  savefig=os.path.join(resfig_dir, "distgraph_broad-scan_spherical-projection.png"),
                                  figsize=(14, 8), aspect="equal")

## |2| Onion-peeled view

In [ ]:
# ==== Define Projection Range ====
depths = np.array([10, 15, 20, 25, 30, 35, 40, 45, 50])  # GASTRULOIDS
depths = np.array([20, 21, 23, 24, 25])
proj_mode = "mean"
layer_thickness = np.round(np.min(img_scale) / 4, 2)
all_min = depths - layer_thickness
all_max = depths + layer_thickness
print(f"Projection depths: {depths} {img_unit}")
all_num = [20 for i in range(len(all_min))]

all_proj = []
all_mesh = []
all_names = [f"{depths[i]}±{layer_thickness}{img_unit}" for i in range(len(depths))]

for i in range(len(depths)):
    print(f">> Projecting at depth {all_names[i]} !")
    proj_broad = analysis.proj2mesh(img=img_raw, mesh=sampl_mesh, unit=img_unit, min_dist_per_vert=None,
                                    scale=img_scale, min_dist=all_min[i], max_dist=all_max[i], num_dist=all_num[i],
                                    mode=proj_mode, show_proj=False, normalise=False)
    sampl_mesh_layer = analysis.scale_mesh(mesh=sampl_mesh, distance=depths[i])
    # sampl_mesh_layer = sampl_mesh.copy()
    all_proj.append(proj_broad)
    all_mesh.append(sampl_mesh_layer)
all_blendings = ["opaque" for i in range(len(depths))]
all_cmaps = ["inferno" for _ in all_proj]
all_colors = [visuals.color_scalar(proj_iter / proj_iter.max(), cmap=all_cmaps[i]) for i, proj_iter in
              enumerate(all_proj)]

In [ ]:
visuals.view_colored_mesh_multiple(mesh_list=all_mesh, vert_colors_list=all_colors, mesh_blending_list=all_blendings,
                                   name_list=all_names)  #, img=img_raw, scale=img_scale)

### |2.1| EMBL Spherical Projection

In [ ]:
sphere_x0, sphere_y0, sphere_z0, sphere_radius = datahandler.load_array("sphere_fit", folderpath=resdata_dir)[0, :]
print(f"sphere_x0 = {sphere_x0} {img_unit}")
print(f"sphere_y0 = {sphere_y0} {img_unit}")
print(f"sphere_z0 = {sphere_z0} {img_unit}")
print(f"sphere_radius = {sphere_radius} {img_unit}")

In [ ]:
min_proj_dist = 2.0
max_proj_dist = 6.0
slice_proj = 0.5
num_proj_samples = int(np.max(img_scale) / np.min(img_scale))
proj_mode = "mean"
min_all = np.arange(min_proj_dist, max_proj_dist, slice_proj)
max_all = min_all + slice_proj
mask = max_all <= max_proj_dist
min_all, max_all = min_all[mask], max_all[mask]
proj_tasks = np.stack([
    min_all,
    max_all,
    np.full(min_all.shape, num_proj_samples)
], axis=1)

proj_radii = sphere_radius - (min_all + 0.5 * slice_proj)
print(f">> Projecting at radii {proj_radii} {img_unit} ...")
# Flip direction of projection for going "inwards"
proj_tasks[:, :2] *= -1

all_projections = np.empty((len(proj_tasks), len(sampl_mesh.vertices)))
for i, proj_task in enumerate(proj_tasks):
    dist_min, dist_max, dist_num = proj_task
    all_projections[i] = analysis.proj2mesh(img=img_raw, mesh=sampl_mesh, unit=img_unit, min_dist_per_vert=None,
                                            scale=img_scale, min_dist=dist_min, max_dist=dist_max, num_dist=dist_num,
                                            mode=proj_mode, show_proj=False, savefig="", normalise=False)
print("=======")
print(f"Projected {len(all_projections)} layers !")

In [ ]:
proj_xcords, proj_ycords = analysis.spherical_project(pts=sampl_mesh.vertices,
                                                      ref_point=[sphere_x0, sphere_y0, sphere_z0])
radial_projection = analysis.create_radial_stack(values=all_projections,
                                                 phi_coords=proj_xcords,
                                                 theta_cords=proj_ycords,
                                                 grid_n=np.mean(img_dim[1:]), projection_radii=proj_radii)
radial_stack, stack_cords = radial_projection
mid_radial_stack_i = radial_stack.shape[0] // 2

visuals.plot_matrix(radial_stack[mid_radial_stack_i], figsize=(14, 8), origin="upper",
                    title=f"R = {proj_radii[mid_radial_stack_i]:,.2f} {img_unit}",
                    savefig=os.path.join(resfig_dir, f"radial-stack_{mid_radial_stack_i}.png"),
                    unit="px", colorbar=True, cmap="inferno")

np.savez_compressed(os.path.join(resdata_dir, "radial_projection.npz"), radial_stack=radial_stack,
                    stack_cords=stack_cords)

In [ ]:
radial_stack = np.load(os.path.join(resdata_dir, "radial_projection.npz"))["radial_stack"]
stack_cords = np.load(os.path.join(resdata_dir, "radial_projection.npz"))["stack_cords"]

datahandler.save_tiff(radial_stack, filepath=os.path.join(resfig_dir, "radial-stack.tiff"))

In [ ]:
visuals.view_colored_mesh_multiple([sampl_mesh, sampl_mesh], vert_colors_list=[
    visuals.color_scalar(analysis.normalise_range(all_projections[0]), cmap="Greens_r"),
    visuals.color_scalar(analysis.normalise_range(all_projections[2]), cmap="Greens_r")])

In [ ]:
# stack_coords_phi = stack_cords[0, ..., 0]
# visuals.plot_matrix(stack_coords_phi, origin="upper", colorbar=True, cmap="coolwarm")
# stack_coords_theta = stack_cords[0, ..., 1]
# visuals.plot_matrix(stack_coords_theta, origin="upper", colorbar=True, cmap="coolwarm")
# stack_coords_radii = stack_cords[:, 0, 0, 2]
# plt.figure()
# plt.hexbin(proj_ycords, proj_xcords, all_projections[0], gridsize=100)
# plt.gca().invert_yaxis()
# plt.gca().invert_xaxis()
# plt.show()

In [ ]:
# ==== Experimental: Spherical Projection relative to Sphere Fit ====
sphere_params_load = datahandler.load_array("sphere_fit", folderpath=resdata_dir)
sphere_x0, sphere_y0, sphere_z0, sphere_radius = sphere_params_load[0, :]
sph_proj_phi, sph_proj_theta = analysis.spherical_project(
    pts=sampl_mesh.vertices, ref_point=[sphere_x0, sphere_y0, sphere_z0])
visuals.plot_spherical_projection(phi=sph_proj_phi, theta=sph_proj_theta, cmap="Greens_r", invert_y_axis=True,
                                  intensities=proj_broad, figsize=(14, 8), ptview=True, aspect="equal",
                                  savefig=os.path.join(resfig_dir, "distgraph_broad-scan_spherical-projection.png"),
                                  ptsize=0.1)
# ==== Experimental: Different Angle Spherical Projection relative to Sphere Fit ====
rotation_angles = [[0, 0, 0],
                   [np.pi / 4, 0, 0],
                   [np.pi / 2, 0, 0],
                   [np.pi / 2, 0, np.pi / 2]]
for rot_angles in rotation_angles:
    print(f"Rotation angles: {np.degrees(rot_angles)}")
    sph_proj_phi, sph_proj_theta = analysis.spherical_project(
        pts=sampl_mesh.vertices, rotate=rot_angles, ref_point=[sphere_x0, sphere_y0, sphere_z0])
    visuals.plot_spherical_projection(phi=sph_proj_phi, theta=sph_proj_theta, intensities=proj_broad,
                                      aspect="equal", ptview=True, figsize=(14, 8))

### |2.2| Check Multi-Layering

In [ ]:
# ==== Define Projection Range and Intermediate Value ====
dist_min = 5
dist_max = 10
dist_middle = dist_min + (dist_max - dist_min) / 2
dist_num = 20
print(f"Chosen Middle Distance {dist_middle} {img_unit}")
# ==== Specify Custom Minimum ====
dist_min_custom = None
# dist_min_custom = np.linspace(0, 5, sampl_mesh.vertices.shape[0])

# ==== Project onto Mesh ====
proj_full = analysis.proj2mesh(img=img_raw, mesh=sampl_mesh, min_dist_per_vert=dist_min_custom,
                               scale=img_scale, min_dist=dist_min, max_dist=dist_max, num_dist=dist_num, mode="max",
                               show_proj=False, return_full=True, unit=img_unit, normalise=False)
# ==== Plot Projected Result ====
distances, dist_points, radial_intensities = proj_full
distcolor, distcmap = visuals.colour_dist(distances=distances, middle_val=dist_max - dist_middle,
                                          radial_points=dist_points, mesh=sampl_mesh,
                                          radial_intensities=radial_intensities)
# visuals.plot_maxproj_pts(verts=sampl_mesh.vertices, colors=distcolor, unit=img_unit, cmap=distcmap, hexsize=300,
#                          savefig=os.path.join(resfig_dir, "maxproj_multi-layer.png"))

In [ ]:
# ==== 3D Render Projected Result ====
visuals.view_colored_mesh(mesh=sampl_mesh, color_override=distcolor)  #, img=img_raw, scale=img_scale)

In [ ]:
# EXPERIMENTAL Ikmi
visuals.view_colored_mesh(mesh=sampl_mesh,
                          color_override=analysis.colour_dist_test(distances=distances, radial_points=dist_points,
                                                                   mesh=sampl_mesh,
                                                                   radial_intensities=radial_intensities))

In [ ]:
# ==== Plot Spherical Projection ====
sph_proj_phi, sph_proj_theta = analysis.spherical_project(
    pts=sampl_mesh.vertices)  #,ref_point=[sphere_x0, sphere_y0, sphere_z0])
visuals.plot_spherical_projection(phi=sph_proj_phi, theta=sph_proj_theta, intensities=distcolor, ptview=True,
                                  savefig=os.path.join(resfig_dir, "multi-layer_spherical-projection.png"),
                                  cmap=distcmap,
                                  figsize=(14, 8))

In [ ]:
# ==== Plot Spherical Projections ====
rotation_angles = [[0, 0, 0],
                   [np.pi / 4, 0, 0],
                   [np.pi / 2, 0, 0],
                   [np.pi / 2, 0, np.pi / 2]]
for rot_angles in rotation_angles:
    print(f"Rotation angles: {np.degrees(rot_angles)}")
    sph_proj_phi, sph_proj_theta = analysis.spherical_project(
        pts=sampl_mesh.vertices, rotate=rot_angles)
    visuals.plot_spherical_projection(phi=sph_proj_phi, theta=sph_proj_theta, intensities=distcolor, ptview=True,
                                      figsize=(5, 5), cmap=distcmap)

## |3| Isolate single layer

In [ ]:
# ==== Projection Logic ====
layer_depth = 20
scale_down_mesh = True
layer_thickness = np.round(np.min(img_scale) / 2, 2)
# layer_thickness = 0.05
dist_min = layer_depth - layer_thickness / 2
dist_max = layer_depth + layer_thickness / 2
proj_mode = "mean"
layer_label = f"proj_{dist_min}_to_{dist_max}_{img_unit}"
dist_middle = dist_min + (dist_max - dist_min) / 2

# ==== Specify Custom Minimum ====
dist_min_custom = None
# dist_min_custom = np.linspace(0, 5, sampl_mesh.vertices.shape[0])
resdata_dir_layer = os.path.join(resdata_dir, layer_label)
resfig_dir_layer = os.path.join(resfig_dir, layer_label)
if not os.path.exists(resdata_dir_layer):
    os.makedirs(resdata_dir_layer)
if not os.path.exists(resfig_dir_layer):
    os.makedirs(resfig_dir_layer)

# ==== Save Sampling Vertices and Normals ====
datahandler.save_array(sampl_mesh.vertices, "verts", header="x,y,z", folderpath=resdata_dir_layer)
datahandler.save_array(sampl_mesh.vertex_normals, "normals", header="nx,ny,nz", folderpath=resdata_dir_layer)
dist_num = 30

# ==== Project onto Mesh ====
proj_layer = analysis.proj2mesh(img=img_raw, mesh=sampl_mesh, min_dist_per_vert=dist_min_custom,
                                scale=img_scale, min_dist=dist_min, max_dist=dist_max,
                                num_dist=dist_num, mode=proj_mode, show_proj=True,
                                savefig=os.path.join(resfig_dir_layer, f"distgraph.png"), unit=img_unit,
                                normalise=False)

# ==== Save Projection ====
if scale_down_mesh:
    layer_mesh = analysis.scale_mesh(mesh=sampl_mesh, distance=dist_middle)
else:
    layer_mesh = sampl_mesh.copy()
datahandler.save_mesh(layer_mesh, filepath=os.path.join(resdata_dir_layer, "layer_mesh.ply"))
datahandler.save_array(proj_layer, "intensities", header="I", folderpath=resdata_dir_layer)

# ==== Plot Projected Result ====
sph_proj_phi, sph_proj_theta = analysis.spherical_project(pts=layer_mesh.vertices)
visuals.plot_spherical_projection(phi=sph_proj_phi, theta=sph_proj_theta, intensities=proj_layer, hexgridsize=400,
                                  savefig=os.path.join(resfig_dir_layer, "spherical-projection.png"), cmap="Greens_r")

In [ ]:
# ==== 3D Render Projected Result ====
visuals.view_colored_mesh(mesh=layer_mesh, vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                            cmap="Greens_r"))  #, img=img_raw, scale=img_scale)

# -- Load Layer --

In [ ]:
layer_names = [
    d for d in os.listdir(resdata_dir)
    if os.path.isdir(os.path.join(resdata_dir, d))
]
print(f"Found layer(s): {layer_names}")

In [ ]:
# ==== Load Projected Result ====
# layer_label = f"proj_{dist_min}_to_{dist_max}_{img_unit}"
layer_label = 'proj_14.95_to_15.05_um'
resdata_dir_layer = os.path.join(resdata_dir, layer_label)
resfig_dir_layer = os.path.join(resfig_dir, layer_label)
proj_layer = datahandler.load_array("intensities", folderpath=resdata_dir_layer)
proj_layer /= proj_layer.max()
try:
    layer_mesh = datahandler.load_mesh(os.path.join(resdata_dir_layer, "layer_mesh.ply"), recalc_normals=False,
                                       clean=False)
except:
    print("Could not find layer mesh, using sampling mesh instead..")
    layer_mesh = datahandler.load_mesh(os.path.join(resdata_dir, "sampling_mesh.ply"), recalc_normals=True, clean=False)

# ==== Plot Projected Result ====
visuals.plot_hist(proj_layer, title="Intensities (a.u.)")
sph_proj_phi, sph_proj_theta = analysis.spherical_project(
    pts=layer_mesh.vertices)  #, ref_point=[sphere_x0, sphere_y0, sphere_z0])
visuals.plot_spherical_projection(phi=sph_proj_phi, theta=sph_proj_theta, intensities=proj_layer, hexgridsize=400,
                                  cmap="Greens_r")

In [ ]:
# ==== 3D Render Projected Result ====
visuals.view_colored_mesh(mesh=layer_mesh, vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                            cmap="Greens_r"))  #img=img_raw,scale=img_scale

# -- Tangential Nematic Analysis --

### |1.1| Select and Define surface patches

In [ ]:
# ==== Select Patch Type and Size ====
patch_avg = ["radius", 20]  # 40
# patch_avg = ["nearest", 1500]
compute_num = 5000  # 5000

# ==== Pres-Select Vertices for 2D+ Orientation Analysis ====
idxs_sel = np.arange(layer_mesh.vertices.shape[0])
idxs_sel = analysis.filter_normal_validity(mesh=layer_mesh, idxs_sel=idxs_sel, k=20, threshold=0.99)

# ==== Filter by Intensity Value ====
cutoff_min_intensity = 0.0 * np.max(proj_layer)
cutoff_max_intensity = 1.0 * np.max(proj_layer)
idxs_sel = idxs_sel[(proj_layer[idxs_sel] > cutoff_min_intensity) & (proj_layer[idxs_sel] < cutoff_max_intensity)]

# ==== Compute only points at Interval ====compute_num
idxs_sel = np.random.choice(idxs_sel, size=int(compute_num))
if len(idxs_sel) == 0:
    print("!! ERROR: No vertices were selected for analysis !!")

# ==== Search Nearest Neighbours ====
patch_type = patch_avg[0]
patch_size = patch_avg[1]

# ==== Tune Plotting parameters ====
if patch_type == "radius":
    idxs_neigh = analysis.coord_search_radius(layer_mesh.vertices, custom_probes=layer_mesh.vertices[idxs_sel],
                                              r=patch_size)
    title_render = f"Extracted directors w.r.t {patch_size}{img_unit}"
elif patch_type == "nearest":
    idxs_neigh = analysis.coord_search_neighbours(layer_mesh.vertices, custom_probes=layer_mesh.vertices[idxs_sel],
                                                  k=patch_size, n_process=8)
    title_render = f"Extracted directors w.r.t {patch_size - 1} neighbours"
else:
    idxs_neigh = patch_label = title_hist = title_render = None
    print(f"[!] Unknown patch type: {patch_type}")

# # ==== Filter Valid Surface Patches ====
# idxs_neigh, valid_patch_idxs = analysis.filter_valid_patches(verts=layer_mesh.vertices, idxs_neigh=idxs_neigh,
#                                                              factor=0.1)
# idxs_sel = idxs_sel[valid_patch_idxs]

# # ==== Filter by Intensity Variation ====
# cutoff_intensity_variance = 0.0
# proj_layer_variance = np.array([np.var(proj_layer[patch_idxs]) for patch_idxs in idxs_neigh])
# visuals.plot_hist(proj_layer_variance, title="Intensity Variance per Patch")
# filter_intens_var_mask = proj_layer_variance > cutoff_intensity_variance

# Filter idxs_sel
# idxs_sel = idxs_sel[filter_intens_var_mask]

# Filter idxs_neigh (keep only the patches that passed)
# idxs_neigh = [patch for i, patch in enumerate(idxs_neigh) if filter_intens_var_mask[i]]

# ==== Find Nearest Intensities ====
proj_layer_neigh = [proj_layer[patch_idxs] for patch_idxs in idxs_neigh]

# ==== Save Vertices for 2D+ Orientation Analysis ====
print(f"Num of directors to be calculated: {len(idxs_sel)} !")
datahandler.save_array(idxs_sel, "calcindeces", header="idx", folderpath=resdata_dir_layer)

# ==== Create the Tangential Bases ====
neighbors_coords = [layer_mesh.vertices[patch] for patch in idxs_neigh]
central_normals = layer_mesh.vertex_normals[idxs_sel]

tan_cords, tan_x, tan_y = analysis.tan_proj(neighbors_coords, central_normals)

# ==== Save the Tangential Bases ====
datahandler.save_array(tan_x, "tan_x", header="t1x,t1y,t1z", folderpath=resdata_dir_layer)
datahandler.save_array(tan_y, "tan_y", header="t2x,t2y,t2z", folderpath=resdata_dir_layer)

In [ ]:
# ==== 3D Render Vertices for 2D+ Orientation Analysis ====
visuals.view_colored_verts(verts=layer_mesh.vertices[idxs_sel],
                           colors=visuals.color_scalar(proj_layer[idxs_sel], cmap="Greens_r"), scale=img_scale)

In [ ]:
# # ==== 3D Render all vertices to be calculated on ====
# all_calculated_mask = np.isin(np.arange(layer_mesh.vertices.shape[0]), idxs_sel)
# all_calculated_color = ["red" if i else "grey" for i in all_calculated_mask]
# visuals.view_colored_mesh(mesh=layer_mesh, vert_colors=all_calculated_color)  #, img=img_raw, scale=img_scale)

### |1.2| Extract Directors

In [ ]:
# ==== Tune Local Orientation Extraction Accuracy ====
# %matplotlib inline
grid_N = 30
box_size = 10
debug_2dcurve_analysis = False
debug_vert_idx, debug_grid_x, debug_grid_y, debug_grid_z = None, None, None, None

print(f">> Using local grid of NxN: {grid_N} and box size: {box_size}...")

if debug_2dcurve_analysis:
    # ==== Pick single vertex for debug ====
    debug_vert_idx = np.random.choice(np.arange(idxs_sel.shape[0]))
    big_grid = analysis.tan_interp_batch(
        coords=[tan_cords[debug_vert_idx]],
        intensities=[proj_layer_neigh[debug_vert_idx]],
        grid_size=grid_N
    )[2][0]
    print(big_grid.shape)
else:
    big_grid = np.vstack([grid.T for grid in analysis.tan_interp_batch(
        coords=tan_cords,
        intensities=proj_layer_neigh,
        grid_size=grid_N
    )[2]])

# ==== Extract Directors ====
directors_2dcurved = analysis.batch_2d_orientation(
    big_grid=big_grid, box_size=box_size,
    vertices=layer_mesh.vertices[idxs_sel],
    tan_x=tan_x, tan_y=tan_y,
    debug=debug_2dcurve_analysis,
    debug_idx=debug_vert_idx,
    debug_line_length=1
)

if not debug_2dcurve_analysis:
    # ==== Save Directors ====
    datahandler.save_array(directors_2dcurved, "directors_2dcurved", header="x,y,z,vx,vy,vz",
                           folderpath=resdata_dir_layer)

    # ==== Plot Directors ====
    visuals.plot_dir_field(directors=directors_2dcurved, title=title_render,
                           savefig=os.path.join(resfig_dir_layer, "directors_2dcurved.png"), veclength=10)

In [ ]:
# ==== 3D Render Directors ====
veclength = 10
vecwidth = 0.5
# visuals.view_3d_vector_field(vec_pos=directors_2dcurved[:, :3], vec_dir=directors_2dcurved[:, 3:], vec_colors="red",
#                              verts=layer_mesh.vertices, verts_colors=visuals.color_scalar(analysis.normalise_range(proj_layer), cmap="Greens_r"),
#                              edge_width=veclength / 6, length=veclength, vec_opacity=0.5, pts_size=1, pts_opacity=0.8,
#                              img=None, scale=img_scale)

# visuals.view_3d_vector_field(vec_pos=directors_2dcurved[:, :3], vec_dir=directors_2dcurved[:, 3:], vec_colors="red",
#                              edge_width=veclength / 6, length=veclength)

# visuals.view_mesh_dir_field([layer_mesh], directors=directors_2dcurved, vec_colors="red",
#                             vec_edge_width=veclength / 6, vec_length=veclength)
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved,
                                    mesh_vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                          cmap="Greys_r"), vec_length=veclength,
                                    vec_edge_width=vecwidth)

In [ ]:
sph_proj_phi, sph_proj_theta = analysis.spherical_project(pts=layer_mesh.vertices)
vec_dir_phi, vec_dir_theta = analysis.spherical_project_vectors(directors_2dcurved[:, :3], directors_2dcurved[:, 3:])

visuals.plot_spherical_projection(
    phi=sph_proj_phi,
    theta=sph_proj_theta,
    intensities=proj_layer,
    vec_pos_phi=sph_proj_phi[idxs_sel],
    vec_pos_theta=sph_proj_theta[idxs_sel],
    vec_dir_phi=vec_dir_phi,
    vec_dir_theta=vec_dir_theta,
    hexgridsize=400,
    scale_factor=2,
    cmap="Greens",
    arrow_alpha=0.7, figsize=(13, 10),
    savefig=os.path.join(resfig_dir_layer, "spherical_projection_extracted-directors.png")
)

### -- Load Directors --

In [ ]:
# ==== Load 2D+ Directors ====
idxs_sel = datahandler.load_array("calcindeces", folderpath=resdata_dir_layer).astype(int)
tan_x = datahandler.load_array("tan_x", folderpath=resdata_dir_layer)
tan_y = datahandler.load_array("tan_y", folderpath=resdata_dir_layer)
directors_2dcurved = datahandler.load_array("directors_2dcurved", folderpath=resdata_dir_layer)

In [ ]:
sph_proj_phi, sph_proj_theta = analysis.spherical_project(pts=layer_mesh.vertices)
vec_dir_phi, vec_dir_theta = analysis.spherical_project_vectors(directors_2dcurved[:, :3], directors_2dcurved[:, 3:])

visuals.plot_spherical_projection(
    phi=sph_proj_phi,
    theta=sph_proj_theta,
    intensities=proj_layer,
    vec_pos_phi=sph_proj_phi[idxs_sel],
    vec_pos_theta=sph_proj_theta[idxs_sel],
    vec_dir_phi=vec_dir_phi,
    vec_dir_theta=vec_dir_theta,
    hexgridsize=400,
    scale_factor=2,
    cmap="Greens",
    arrow_alpha=0.7, figsize=(13, 10)
)

In [ ]:
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved, vec_edge_width=0.3, vec_length=5,
                                    mesh_vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                          cmap="Greys_r"))

### |2.1| Remove Initial Noise by Nematic Averaging

In [ ]:
# ==== Tune Curved Nematic Analysis Number of Neighbours or Radius ====
patch_avg = ["radius", 60]  # 40
# patch_avg = ["nearest", 200]
patch_type = patch_avg[0]
patch_size = patch_avg[1]

# ==== Tune Plotting parameters ====
vec_length = 10
vec_edge_width = vec_length / 6
plot2d_view = (20, 0)
renderfigsize = (6, 5)
histfigsize = (4, 3)
veccoords = directors_2dcurved[:, :3]
if patch_type == "radius":
    neigh_idxs = analysis.coord_search_radius(veccoords, r=patch_size)
    patch_label = f"r-{patch_size}{img_unit}"
    title_hist = f"Avg over {patch_size}{img_unit}: Order Scalar $S$"
    title_render = f"Avg over {patch_size}{img_unit}: Average Directors"
elif patch_type == "nearest":
    neigh_idxs = analysis.coord_search_neighbours(veccoords, k=patch_size, n_process=8)
    patch_label = f"k-{patch_size}"
    title_hist = f"Avg over {patch_size - 1} neighbours: Order Scalar $S$"
    title_render = f"Avg over {patch_size - 1} neighbours: Average Directors"
else:
    neigh_idxs = patch_label = title_hist = title_render = None
    print(f"[!] Unknown patch type: {patch_type}")

# ==== Visualise Averaging Patch ====
# patch_sel_idx = np.random.choice(range(len(neigh_idxs)))
# patch_color = np.array(["#FF0000" for _ in range(len(veccoords))])
# patch_color[neigh_idxs[patch_sel_idx]] = "#FFFF00"
# patch_color[neigh_idxs[patch_sel_idx][0]] = "#0000FF"
# visuals.view_colored_verts(verts=veccoords, colors=list(patch_color), use_orig_color=True)

# ==== Calculate Curved Nematic Order ====
S_2dcurv, n_avg_2dcurv = analysis.avg_tan_nem_tens(t1_cov=tan_x, t2_cov=tan_y, directors=directors_2dcurved,
                                                   neigh_idxs=neigh_idxs)

# ==== Save Curved Nematic Order ====
datahandler.save_array(S_2dcurv, name=f"S-order-init_2dcurved_{patch_label}", header="S", folderpath=resdata_dir_layer)
datahandler.save_array(np.column_stack((veccoords, n_avg_2dcurv)), name=f"directors-avg-init_2dcurved_{patch_label}",
                       header="x,y,z,vx,vy,vz", folderpath=resdata_dir_layer)

# ==== Plot Curved Nematic Order ====
directors_2dcurved_avg = directors_2dcurved.copy()
directors_2dcurved_avg[:, 3:] = n_avg_2dcurv
savefig_render = os.path.join(resfig_dir_layer, f"field_intial-avg-nematic_{patch_label}.png")
savefig_hist = os.path.join(resfig_dir_layer, f"hist_intial-order-s_{patch_label}.png")

visuals.plot_dir_field(directors=directors_2dcurved_avg, veclength=vec_length, view_init=plot2d_view, veccolor=S_2dcurv,
                       cmap_label="order scalar $S$", title=title_render, manual_vminmax=[0, 1],
                       savefig=savefig_render, figsize=renderfigsize, show_axes=False)
visuals.plot_hist(array=S_2dcurv, title=title_hist, savefig=savefig_hist,
                  figsize=histfigsize, xlim=[0, 1])
directors_2dcurved_avg_init = directors_2dcurved_avg.copy()

In [ ]:
# ==== 3D Render Curved Nematic Order ====
vec_length = 10
vec_edge_width = 0.5
vec_length = 20
vec_edge_width = 1
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg,
                                    vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]),
                                    mesh_vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                          cmap="Greys_r"),
                                    vec_length=0.5 * vec_length, vec_edge_width=vec_edge_width)

# visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg,
#                                     vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]),
#                                     mesh_vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
#                                                                           cmap="Greys_r"),
#                                     vec_length=vec_length, vec_edge_width=vec_edge_width)
# visuals.view_3d_vector_field(vec_pos=plot_vec_posdir[:, :3], vec_dir=plot_vec_posdir[:, 3:],
#                              vec_colors=visuals.color_scalar(S_2dcurv, cmap="Spectral", manual_vminmax=[0, 1]),
#                              length=vec_length, pts_size=1,
#                              edge_width=vec_edge_width)

# visuals.view_3d_vector_field(vec_pos=plot_vec_posdir[:, :3], vec_dir=plot_vec_posdir[:, 3:],
#                              vec_colors=visuals.color_scalar(S_2dcurv, cmap="Spectral"),
#                              length=vec_length, edge_width=vec_edge_width,
#                              verts=layer_mesh.vertices,
#                              verts_colors=visuals.color_scalar(analysis.normalise_range(proj_layer), cmap="Greens_r"))


In [ ]:
sph_proj_phi, sph_proj_theta = analysis.spherical_project(pts=layer_mesh.vertices)
vec_dir_phi, vec_dir_theta = analysis.spherical_project_vectors(directors_2dcurved_avg[:, :3],
                                                                directors_2dcurved_avg[:, 3:])

visuals.plot_spherical_projection(
    phi=sph_proj_phi,
    theta=sph_proj_theta,
    intensities=proj_layer,
    vec_pos_phi=sph_proj_phi[idxs_sel],
    vec_pos_theta=sph_proj_theta[idxs_sel],
    vec_dir_phi=vec_dir_phi,
    vec_dir_theta=vec_dir_theta,
    vec_cmap_label="order scalar $S$",
    hexgridsize=400,
    scale_factor=2,
    cmap="Greys_r", vec_width=0.0015,
    veccolor=S_2dcurv,
    vec_manual_vminmax=[0, 1],
    arrow_alpha=1.0, figsize=(20, 13)
)

In [ ]:
thickness_vals = datahandler.load_array(name="thickness", folderpath=resdata_dir)

In [ ]:
layer_mesh_1 = datahandler.load_mesh(
    filepath='/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!nemo/claire_bounddefects/t=0_c=0/data/proj_13.5_to_16.5_um_inner/layer_mesh.ply',
    recalc_normals=True, clean=False)
layer_mesh_2 = datahandler.load_mesh(
    filepath='/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!nemo/claire_bounddefects/t=0_c=0/data/proj_3.5_to_6.5_um_outer/layer_mesh.ply',
    recalc_normals=True, clean=False)

In [ ]:
rot_angles = [1, 0, 0]
sph_proj_phi_1, sph_proj_theta_1 = analysis.spherical_project(pts=layer_mesh_1.vertices, rotate=rot_angles)
sph_proj_phi_2, sph_proj_theta_2 = analysis.spherical_project(pts=layer_mesh_2.vertices, rotate=rot_angles)
vec_dir_phi, vec_dir_theta = analysis.spherical_project_vectors(directors_2dcurved_avg[:, :3],
                                                                directors_2dcurved_avg[:, 3:], rotate=rot_angles)

visuals.plot_spherical_projection(
    phi=sph_proj_phi_1,
    theta=sph_proj_theta_1,
    intensities=thickness_vals,
    vec_pos_phi=sph_proj_phi_2[idxs_sel],
    vec_pos_theta=sph_proj_theta_2[idxs_sel],
    vec_dir_phi=vec_dir_phi,
    vec_dir_theta=vec_dir_theta,
    vec_cmap_label="order scalar $S$",
    hexgridsize=400,
    scale_factor=2, alpha=0.3, cmap_label=f"Thickness ({img_unit})",
    cmap="inferno", vec_width=0.0015,
    veccolor=S_2dcurv,
    vec_manual_vminmax=[0, 1], manual_vminmax=[thickness_vals.min(), thickness_vals.max()],
    arrow_alpha=1.0, figsize=(20, 13),
    savefig=os.path.join(resfig_dir_layer, f"spherical_projection_field_avg-nematic_thickness.png")
)

### |2.2| Compute nematic order scalar S

In [ ]:
# ==== Tune Curved Nematic Analysis Number of Neighbours ====
patch_avg = ["radius", 30]
# patch_avg = ["nearest", 200]
patch_type = patch_avg[0]
patch_size = patch_avg[1]

# ==== Tune Plotting parameters ====
vec_length = 20
vec_edge_width = vec_length / 6
plot2d_view = (20, 0)
renderfigsize = (6, 5)
histfigsize = (4, 3)

veccoords = directors_2dcurved_avg_init[:, :3]
if patch_type == "radius":
    neigh_idxs = analysis.coord_search_radius(veccoords, r=patch_size)
    patch_label = f"r-{patch_size}{img_unit}"
    title_hist = f"Avg over {patch_size}{img_unit}: Order Scalar $S$"
    title_render = f"Avg over {patch_size}{img_unit}: Average Directors"
elif patch_type == "nearest":
    neigh_idxs = analysis.coord_search_neighbours(veccoords, k=patch_size, n_process=8)
    patch_label = f"k-{patch_size}"
    title_hist = f"Avg over {patch_size - 1} neighbours: Order Scalar $S$"
    title_render = f"Avg over {patch_size - 1} neighbours: Average Directors"
else:
    neigh_idxs = patch_label = title_hist = title_render = None
    print(f"[!] Unknown patch type: {patch_type}")

# ==== Visualise Averaging Patch ====
# patch_sel_idx = np.random.choice(range(len(neigh_idxs)))
# patch_color = np.array(["#FF0000" for _ in range(len(veccoords))])
# patch_color[neigh_idxs[patch_sel_idx]] = "#FFFF00"
# patch_color[neigh_idxs[patch_sel_idx][0]] = "#0000FF"
# visuals.view_colored_verts(verts=veccoords, colors=list(patch_color), use_orig_color=True)

# ==== Calculate Curved Nematic Order ====
S_2dcurv, n_avg_2dcurv = analysis.avg_tan_nem_tens(t1_cov=tan_x, t2_cov=tan_y, directors=directors_2dcurved_avg_init,
                                                   neigh_idxs=neigh_idxs)

# ==== Save Curved Nematic Order ====
datahandler.save_array(S_2dcurv, name=f"S-order_2dcurved_{patch_label}", header="S", folderpath=resdata_dir_layer)
datahandler.save_array(np.column_stack((veccoords, n_avg_2dcurv)), name=f"directors-avg_2dcurved_{patch_label}",
                       header="x,y,z,vx,vy,vz", folderpath=resdata_dir_layer)

# ==== Plot Curved Nematic Order ====
directors_2dcurved_avg = directors_2dcurved_avg_init.copy()
directors_2dcurved_avg[:, 3:] = n_avg_2dcurv
savefig_render = os.path.join(resfig_dir_layer, f"field_avg-nematic_{patch_label}.png")
savefig_hist = os.path.join(resfig_dir_layer, f"hist_order-s_{patch_label}.png")

visuals.plot_dir_field(directors=directors_2dcurved_avg, veclength=vec_length, view_init=plot2d_view, veccolor=S_2dcurv,
                       cmap_label="order scalar $S$", title=title_render, manual_vminmax=[0, 1],
                       savefig=savefig_render, figsize=renderfigsize, show_axes=False)
visuals.plot_hist(array=S_2dcurv, title=title_hist, savefig=savefig_hist,
                  figsize=histfigsize, xlim=[0, 1])

In [ ]:
sph_proj_phi, sph_proj_theta = analysis.spherical_project(pts=layer_mesh.vertices)
vec_dir_phi, vec_dir_theta = analysis.spherical_project_vectors(directors_2dcurved_avg[:, :3],
                                                                directors_2dcurved_avg[:, 3:])

visuals.plot_spherical_projection(
    phi=sph_proj_phi,
    theta=sph_proj_theta,
    intensities=proj_layer,
    vec_pos_phi=sph_proj_phi[idxs_sel],
    vec_pos_theta=sph_proj_theta[idxs_sel],
    vec_dir_phi=vec_dir_phi,
    vec_dir_theta=vec_dir_theta,
    vec_cmap_label="order scalar $S$",
    hexgridsize=400,
    scale_factor=5,
    cmap="Greys_r", vec_width=0.0015,
    veccolor=S_2dcurv,
    vec_manual_vminmax=[0, 1],
    arrow_alpha=1.0, figsize=(20, 13),
    savefig=os.path.join(resfig_dir_layer, f"spherical_projection_field_avg-nematic_{patch_label}.png"),
)

In [ ]:
# ==== 3D Render Curved Nematic Order ====
vec_length = 10
vec_edge_width = 1
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg,
                                    vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]),
                                    mesh_vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                          cmap="Greys_r"),
                                    vec_length=vec_length, vec_edge_width=vec_edge_width)
# visuals.view_3d_vector_field(vec_pos=directors_2dcurved_avg[:, :3], vec_dir=directors_2dcurved_avg[:, 3:],
#                              vec_colors=visuals.color_scalar(S_2dcurv, cmap="Spectral", manual_vminmax=[0, 1]),
#                              length=vec_length, pts_size=1,
#                              edge_width=vec_edge_width)
#
# visuals.view_3d_vector_field(vec_pos=directors_2dcurved_avg[:, :3], vec_dir=directors_2dcurved_avg[:, 3:],
#                              vec_colors=visuals.color_scalar(S_2dcurv, cmap="Spectral"),
#                              length=vec_length, edge_width=vec_edge_width,
#                              verts=layer_mesh.vertices,
#                              verts_colors=visuals.color_scalar(analysis.normalise_range(proj_layer), cmap="Greens_r"))


### -- Load 2D+ Order --

In [ ]:
# ==== Load 2D+ nematic order ====
patch_avg = ["radius", 40]
# patch_avg = ["nearest", 200]
patch_type = patch_avg[0]
patch_size = patch_avg[1]

if patch_type == "radius":
    patch_label = f"r-{patch_size}{img_unit}"
    title_hist = f"Avg over {patch_size}{img_unit}: Order Scalar $S$"
    title_render = f"Avg over {patch_size}{img_unit}: Average Directors"
elif patch_type == "nearest":
    patch_label = f"k-{patch_size}"
    title_hist = f"Avg over {patch_size - 1} neighbours: Order Scalar $S$"
    title_render = f"Avg over {patch_size - 1} neighbours: Average Directors"
else:
    neigh_idxs = patch_label = title_hist = title_render = None
    print(f"[!] Unknown patch type: {patch_type}")

load_label_inset = "-init"
directors_2dcurved_avg = datahandler.load_array(f"directors-avg{load_label_inset}_2dcurved_{patch_label}",
                                                folderpath=resdata_dir_layer)
S_2dcurv = datahandler.load_array(f"S-order{load_label_inset}_2dcurved_{patch_label}", folderpath=resdata_dir_layer)
visuals.plot_hist(S_2dcurv, title=title_hist, xlim=[0, 1])
visuals.plot_dir_field(directors=directors_2dcurved_avg, veccolor=S_2dcurv, veclength=4, view_init=(20, 0),
                       cmap_label="order scalar $S$", title=title_render, manual_vminmax=[0, 1], show_axes=False)
sph_proj_phi, sph_proj_theta = analysis.spherical_project(pts=layer_mesh.vertices)
vec_dir_phi, vec_dir_theta = analysis.spherical_project_vectors(directors_2dcurved_avg[:, :3],
                                                                directors_2dcurved_avg[:, 3:])

visuals.plot_spherical_projection(
    phi=sph_proj_phi,
    theta=sph_proj_theta,
    intensities=proj_layer,
    vec_pos_phi=sph_proj_phi[idxs_sel],
    vec_pos_theta=sph_proj_theta[idxs_sel],
    vec_dir_phi=vec_dir_phi,
    vec_dir_theta=vec_dir_theta,
    vec_cmap_label="order scalar $S$",
    hexgridsize=400,
    scale_factor=2,
    cmap="Greys_r", vec_width=0.0015,
    veccolor=S_2dcurv,
    vec_manual_vminmax=[0, 1],
    arrow_alpha=1.0, figsize=(20, 13)
)

In [ ]:
vec_length = 10
vec_edge_width = 3
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg,
                                    vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]),
                                    mesh_vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                          cmap="Greys_r"),
                                    vec_length=vec_length, vec_edge_width=0.2)

In [ ]:
# ==== 3D Render Results ====
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg,
                                    vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]),
                                    mesh_vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                          cmap="Greens_r"), vec_length=vec_length,
                                    vec_edge_width=vec_edge_width)

In [ ]:
# ==== 3D Render Results ====
visuals.view_colored_mesh(layer_mesh,
                          visuals.color_scalar(analysis.interpolate_on_mesh(layer_mesh, idxs_sel, S_2dcurv, k=10),
                                               manual_vminmax=[0, 1]))

In [ ]:
# ==== 3D Render Results ====
visuals.view_colored_mesh_multiple([layer_mesh, layer_mesh],
                                   [visuals.color_scalar(
                                       analysis.interpolate_on_mesh(layer_mesh, idxs_sel, S_2dcurv, k=30),
                                       manual_vminmax=[0, 1]),
                                       visuals.color_scalar(proj_layer, normalise=True, cmap="Greens_r")])

### |3| Identify Defect Location(s) + Geodesic Distance

In [ ]:
# ==== Find Defects and Inter ====
dist_cutoff_defect_localisation = 100
max_candidates_defect_localisation = 30
defect_idxs, rel_dists = analysis.select_geodesic_defects(S_2dcurv, layer_mesh, idxs_sel,
                                                          dist_cutoff=dist_cutoff_defect_localisation, unit=img_unit,
                                                          max_candidates=max_candidates_defect_localisation)
datahandler.save_array(defect_idxs, name="defect-idxs", header="idx", folderpath=resdata_dir_layer)

visuals.plot_dir_field(directors=directors_2dcurved_avg, veclength=1, veccolor="red",
                       marker=directors_2dcurved_avg[:, :3][defect_idxs],
                       pt_label="Defect Locations", vec_alpha=0.5, freq=5,
                       pt_color=visuals.color_scalar(np.linspace(0, 1, len(defect_idxs)), "Set1"),
                       pt_alpha=1.0, cmap_label="order parameter $S$", manual_vminmax=[0, 1],
                       savefig=os.path.join(resfig_dir_layer, f"defect-locations.png"))

In [ ]:
sph_proj_phi, sph_proj_theta = analysis.spherical_project(pts=layer_mesh.vertices)
vec_dir_phi, vec_dir_theta = analysis.spherical_project_vectors(directors_2dcurved_avg[:, :3],
                                                                directors_2dcurved_avg[:, 3:])
visuals.plot_spherical_projection(
    phi=sph_proj_phi,
    theta=sph_proj_theta,
    intensities=proj_layer,
    vec_pos_phi=sph_proj_phi[idxs_sel],
    vec_pos_theta=sph_proj_theta[idxs_sel],
    vec_dir_phi=vec_dir_phi,
    vec_dir_theta=vec_dir_theta,
    vec_cmap_label="order scalar $S$",
    hexgridsize=400, scale_factor=2,
    cmap="Greys_r", vec_width=0.0015,
    veccolor=S_2dcurv, vec_manual_vminmax=[0, 1],
    arrow_alpha=1.0, figsize=(20, 13),
    marker_idxs=idxs_sel[defect_idxs],
    marker_color=visuals.color_scalar(np.linspace(0, 1, len(defect_idxs)), "Set1"),
    savefig=os.path.join(resfig_dir_layer, f"defect-locations.png")
)

In [ ]:
# ==== 3D Render Results ====
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved, vec_edge_width=0.1,
                                    vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]), marker_size=500,
                                    mesh_vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                          cmap="Greys"),
                                    markers=layer_mesh.vertices[idxs_sel[defect_idxs]],
                                    marker_colors=visuals.color_scalar(np.linspace(0, 1, len(defect_idxs)), "Set1"))

In [ ]:
# ==== Calculate Geodesic Distance between Defects ====
defect_1_index = idxs_sel[defect_idxs[0]]
defect_2_index = idxs_sel[defect_idxs[1]]
dist = analysis.geodesic_distmesh(mesh=layer_mesh, index1=defect_1_index, index2=defect_2_index, debug=True)
visuals.plot_dir_field(directors=directors_2dcurved_avg, veclength=0.3,
                       veccolor="red", marker=layer_mesh.vertices[[defect_1_index, defect_2_index]],
                       pt_label="Points", pt_alpha=1.0, view_init=[90, 0])

### -- Load Defect(s) Position(s) --

In [ ]:
defect_idxs = datahandler.load_array(name="defect-idxs", folderpath=resdata_dir_layer).astype(int)
print(f"Found {len(defect_idxs)} defect indeces {defect_idxs}!")
visuals.plot_dir_field(directors=directors_2dcurved_avg, veclength=1, veccolor="red",
                       marker=directors_2dcurved_avg[:, :3][defect_idxs],
                       pt_label="Defect Locations", vec_alpha=0.5, freq=5,
                       pt_color=visuals.color_scalar(np.linspace(0, 1, len(defect_idxs)), "Set1"),
                       pt_alpha=1.0, cmap_label="order parameter $S$", manual_vminmax=[0, 1])
sph_proj_phi, sph_proj_theta = analysis.spherical_project(pts=layer_mesh.vertices)
vec_dir_phi, vec_dir_theta = analysis.spherical_project_vectors(directors_2dcurved_avg[:, :3],
                                                                directors_2dcurved_avg[:, 3:])
visuals.plot_spherical_projection(
    phi=sph_proj_phi,
    theta=sph_proj_theta,
    intensities=proj_layer,
    vec_pos_phi=sph_proj_phi[idxs_sel],
    vec_pos_theta=sph_proj_theta[idxs_sel],
    vec_dir_phi=vec_dir_phi,
    vec_dir_theta=vec_dir_theta,
    vec_cmap_label="order scalar $S$",
    hexgridsize=400,
    scale_factor=2,
    cmap="Greys_r", vec_width=0.0015,
    veccolor=S_2dcurv,
    vec_manual_vminmax=[0, 1],
    arrow_alpha=1.0, figsize=(20, 13),
    marker_idxs=idxs_sel[defect_idxs],
    marker_color=visuals.color_scalar(np.linspace(0, 1, len(defect_idxs)), "Set1")
)

In [ ]:
# ==== 3D Render Results ====
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg,
                                    vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]), marker_size=500,
                                    mesh_vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                          cmap="Greys"),
                                    markers=layer_mesh.vertices[idxs_sel[defect_idxs]],
                                    marker_colors=visuals.color_scalar(np.linspace(0, 1, len(defect_idxs)), "Set1"))

### |4| Topological Charge of Nematic Point Defects

In [ ]:
# # ==== Calculate Gaussian Curvature for Topological Charge Analysis ====
gauss_exp = 1 / (np.ptp(layer_mesh.vertices, axis=0).mean() / 2) ** 2
gauss_order_min, gauss_order_max = round(np.log10(gauss_exp)) - 1, round(np.log10(gauss_exp)) + 1
print(f"Gauss should be around {gauss_exp:.2e} (1/{img_unit}^2)")
gauss_crop_range = [0.01 * 10 ** gauss_order_min, 10 ** gauss_order_max]
gauss_crop_range = None

patch_info = ["radius", 20]
# patch_info = ["nearest", 20]


curv_charge_quick = analysis.curvature_by_srf_fit(mesh=layer_mesh, patch_mode=patch_info[0], patch_size=patch_info[1],
                                                  debug=True, gauss_crop_range=gauss_crop_range, num_sample=3000,
                                                  filter_boundary=False, boundary_excl_factor=0.0)
# Results given as curv_charge_quick = C_gauss, C_mean, Gauss_idxs, mean_idxs
gauss_curv_smooth = analysis.interpolate_on_mesh(mesh=layer_mesh, value_idxs=curv_charge_quick[2],
                                                 values=curv_charge_quick[0], k=10)

In [ ]:
visuals.view_colored_mesh(layer_mesh, visuals.color_scalar(gauss_curv_smooth, normalise=True))

In [ ]:
# ==== Calculate Curved Topological Charge ====
patch_charge = ["radius", 70]
# patch_charge = ["nearest", 1000]
patch_type = patch_charge[0]
patch_size = patch_charge[1]

if patch_type == "radius":
    patch_label = f"r-{patch_size}{img_unit}"
    charge_patch_idxs = analysis.coord_search_radius(layer_mesh.vertices,
                                                     custom_probes=layer_mesh.vertices[idxs_sel[defect_idxs]],
                                                     r=patch_size)
elif patch_type == "nearest":
    patch_label = f"k-{patch_size}"
    charge_patch_idxs = analysis.coord_search_neighbours(layer_mesh.vertices,
                                                         custom_probes=layer_mesh.vertices[idxs_sel[defect_idxs]],
                                                         k=patch_size)
else:
    charge_patch_idxs = None
    print(f"[!] Unknown patch type: {patch_type}")

# charge_patch_idxs, _ = analysis.filter_valid_patches(layer_mesh.vertices, charge_patch_idxs, factor=0.2)
# charge_patch_idxs = analysis.unique_neighborhoods(charge_patch_idxs)
defect_idxs_calc = [np.flatnonzero(idxs_sel == lst[0])[0]
                    for lst in charge_patch_idxs
                    if np.any(idxs_sel == lst[0])]

_, tan_x_all, tan_y_all = analysis.tan_proj(layer_mesh.vertices[:, np.newaxis], layer_mesh.vertex_normals)
m_charge, calc_charge_loop_idxs = analysis.curved_nem_charge(mesh=layer_mesh, directors=directors_2dcurved_avg,
                                                             calc_idxs=defect_idxs_calc,
                                                             director_indeces=idxs_sel,
                                                             tan_x=tan_x_all, tan_y=tan_y_all,
                                                             c_gauss=gauss_curv_smooth,
                                                             loop_angle_precision=1, patch_mode=patch_type,
                                                             patch_size=patch_size, debug=True,
                                                             correct_orientation=True)
print(f"Final number of defects: {len(m_charge)} !")
visuals.plot_dir_field(directors=directors_2dcurved_avg, veclength=0.02, veccolor=S_2dcurv,
                       marker=np.vstack([layer_mesh.vertices[i] for i in calc_charge_loop_idxs]),
                       pt_label="Charge Calculation Line",
                       pt_alpha=1.0, cmap_label="order parameter $S$", manual_vminmax=[0, 1],
                       savefig=os.path.join(resfig_dir_layer, f"top-charge-loops_{patch_label}.png"))

m_charge_extended = np.full(len(layer_mesh.vertices), 0.0)
if patch_type == "radius":
    charge_patch_idxs_calc = analysis.coord_search_radius(layer_mesh.vertices,
                                                          custom_probes=layer_mesh.vertices[idxs_sel[defect_idxs_calc]],
                                                          r=patch_size)
elif patch_type == "nearest":
    charge_patch_idxs_calc = analysis.coord_search_neighbours(layer_mesh.vertices,
                                                              custom_probes=layer_mesh.vertices[
                                                                  idxs_sel[defect_idxs_calc]],
                                                              k=patch_size)
else:
    charge_patch_idxs_calc = None
    print(f"[!] Unknown patch type: {patch_type}")
if isinstance(charge_patch_idxs_calc, list):
    flat_idxs = np.concatenate([np.atleast_1d(np.array(x)) for x in charge_patch_idxs_calc if len(x) > 0])
else:
    flat_idxs = np.ravel(charge_patch_idxs_calc)

m_charge_extended[flat_idxs] = np.repeat(m_charge, [len(np.atleast_1d(x)) for x in charge_patch_idxs_calc])
m_charge_extended = m_charge_extended.ravel()

visuals.plot_dir_field(directors=directors_2dcurved_avg, veclength=10, veccolor=m_charge_extended[idxs_sel],
                       cmap="rainbow",
                       title=r"TOTAL CHARGE$\approx$" + f"{np.nansum(m_charge):.3}",
                       cmap_label="topological charge $m$",
                       manual_vminmax=[-1, 1], show_axes=False, marker=layer_mesh.vertices[idxs_sel[defect_idxs_calc]],
                       savefig=os.path.join(resfig_dir_layer, f"top-charge-shaded_{patch_label}.png"))

# ==== Save Curved Topological Charge ====
datahandler.save_array(m_charge, name=f"top-charge_2dcurved_{patch_label}", header="m", folderpath=resdata_dir_layer)
datahandler.save_array(defect_idxs_calc, name=f"top-charge_2dcurved_{patch_label}_idxs", header="idx",
                       folderpath=resdata_dir_layer)

# ==== Plot Curved Topological Charge ====
visuals.plot_hist(m_charge, title=f"Sum(m)={np.nansum(m_charge)}",
                  savefig=os.path.join(resfig_dir_layer, f"hist_top-charge_{patch_label}"))



In [ ]:
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg, mesh_shading="smooth",
                                    mesh_vert_colors="grey",
                                    vec_colors=visuals.color_scalar(m_charge_extended[idxs_sel], manual_vminmax=[-1, 1],
                                                                    cmap="rainbow"), vec_edge_width=0.05,
                                    markers=layer_mesh.vertices[idxs_sel[defect_idxs_calc]],
                                    marker_colors=visuals.color_scalar(m_charge, manual_vminmax=[-1, 1],
                                                                       cmap="rainbow"), marker_size=400)

In [ ]:
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg, mesh_shading="smooth",
                                    mesh_vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                          cmap="Greens_r"),
                                    vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]),
                                    markers=layer_mesh.vertices[idxs_sel[defect_idxs_calc]], vec_edge_width=0.2,
                                    vec_length=10,
                                    marker_colors=visuals.color_scalar(m_charge, manual_vminmax=[-1, 1],
                                                                       cmap="rainbow"), marker_size=400)

In [ ]:
sph_proj_phi, sph_proj_theta = analysis.spherical_project(pts=layer_mesh.vertices)
vec_dir_phi, vec_dir_theta = analysis.spherical_project_vectors(directors_2dcurved_avg[:, :3],
                                                                directors_2dcurved_avg[:, 3:])
visuals.plot_spherical_projection(
    phi=sph_proj_phi,
    theta=sph_proj_theta,
    intensities=proj_layer,
    vec_pos_phi=sph_proj_phi[idxs_sel],
    vec_pos_theta=sph_proj_theta[idxs_sel],
    vec_dir_phi=vec_dir_phi,
    vec_dir_theta=vec_dir_theta,
    vec_cmap_label="order scalar $S$",
    hexgridsize=400,
    scale_factor=2,
    cmap="Greys_r", vec_width=0.0015,
    veccolor=S_2dcurv,
    vec_manual_vminmax=[0, 1],
    arrow_alpha=1.0, figsize=(20, 13),
    marker_idxs=idxs_sel[defect_idxs_calc],
    marker_color=visuals.color_scalar(m_charge, manual_vminmax=[-1, 1], cmap="rainbow"),
    savefig=os.path.join(resfig_dir_layer, f"defect-charges.png")
)

### -- Load Defect(s) Charge(s) --

In [ ]:
patch_charge = ["radius", 40]
# patch_charge = ["nearest", 200]
patch_type = patch_charge[0]
patch_size = patch_charge[1]

if patch_type == "radius":
    patch_label = f"r-{patch_size}{img_unit}"
elif patch_type == "nearest":
    patch_label = f"k-{patch_size}"
else:
    print(f"[!] Unknown patch type: {patch_type}")

m_charge = datahandler.load_array(name=f"top-charge_2dcurved_{patch_label}", folderpath=resdata_dir_layer)
sph_proj_phi, sph_proj_theta = analysis.spherical_project(pts=layer_mesh.vertices)
vec_dir_phi, vec_dir_theta = analysis.spherical_project_vectors(directors_2dcurved_avg[:, :3],
                                                                directors_2dcurved_avg[:, 3:])
defect_idxs_calc = datahandler.load_array(name=f"top-charge_2dcurved_{patch_label}_idxs",
                                          folderpath=resdata_dir_layer).astype(int)
visuals.plot_spherical_projection(
    phi=sph_proj_phi,
    theta=sph_proj_theta,
    intensities=proj_layer,
    vec_pos_phi=sph_proj_phi[idxs_sel],
    vec_pos_theta=sph_proj_theta[idxs_sel],
    vec_dir_phi=vec_dir_phi,
    vec_dir_theta=vec_dir_theta,
    vec_cmap_label="order scalar $S$",
    hexgridsize=400,
    scale_factor=2,
    cmap="Greys_r", vec_width=0.0015,
    veccolor=S_2dcurv,
    vec_manual_vminmax=[0, 1],
    arrow_alpha=1.0, figsize=(20, 13),
    marker_idxs=idxs_sel[defect_idxs_calc],
    marker_color=visuals.color_scalar(m_charge, manual_vminmax=[-1, 1], cmap="rainbow"),
    savefig=os.path.join(resfig_dir_layer, f"defect-charges.png")
)

In [ ]:
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg, mesh_shading="smooth",
                                    mesh_vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                          cmap="Greens_r"),
                                    vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]),
                                    markers=layer_mesh.vertices[idxs_sel[defect_idxs_calc]], vec_edge_width=0.2,
                                    vec_length=10,
                                    marker_colors=visuals.color_scalar(m_charge, manual_vminmax=[-1, 1],
                                                                       cmap="rainbow"), marker_size=400)

### (experimental) Defect Polarisation

In [ ]:
patch_polarisation = ["radius", 100]
# patch_polarisation = ["nearest", 80]
patch_type = patch_polarisation[0]
patch_size = patch_polarisation[1]

if patch_type == "radius":
    patch_label = f"r-{patch_size}{img_unit}"
elif patch_type == "nearest":
    patch_label = f"k-{patch_size}"
else:
    print(f"[!] Unknown patch type: {patch_type}")
from scripts import extra_nematic

pol_vecfield, pol_idxs = extra_nematic.compute_defect_polarisations(
    mesh=layer_mesh,
    idxs_sel=idxs_sel,
    directors=directors_2dcurved_avg,
    vertex_normals=layer_mesh.vertex_normals.copy(),
    defect_idxs_calc=defect_idxs_calc,
    m_charge=np.round(m_charge, 2),
    patch_type=patch_type,
    patch_size=patch_size, show_profile=True
)
charge_pol_linked_idxs = np.argsort(defect_idxs_calc)[
    np.searchsorted(defect_idxs_calc, pol_idxs, sorter=np.argsort(defect_idxs_calc))]

datahandler.save_array(pol_vecfield, name=f"def-pol_2dcurved_{patch_label}", header="x,y,z,vx,vy,vz",
                       folderpath=resdata_dir_layer)
datahandler.save_array(charge_pol_linked_idxs, name=f"def-pol_2dcurved_{patch_label}_idxs", header="idx",
                       folderpath=resdata_dir_layer)

In [ ]:
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg, mesh_shading="smooth",
                                    mesh_vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                          cmap="Greys"),
                                    vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]),
                                    markers=layer_mesh.vertices[idxs_sel[defect_idxs_calc]],
                                    marker_colors=visuals.color_scalar(m_charge, manual_vminmax=[-1, 1],
                                                                       cmap="rainbow"), marker_size=400,
                                    marker_vectors=pol_vecfield, marker_vectors_length=10,
                                    marker_vector_width=1, vec_edge_width=0.05,
                                    marker_vectors_color=visuals.color_scalar(m_charge[charge_pol_linked_idxs],
                                                                              manual_vminmax=[-1, 1],
                                                                              cmap="rainbow"), )

In [ ]:
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg, mesh_shading="smooth",
                                    mesh_vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                          cmap="Greys"),
                                    vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]),
                                    markers=layer_mesh.vertices[idxs_sel[defect_idxs_calc]],
                                    marker_colors=visuals.color_scalar(m_charge, manual_vminmax=[-1, 1],
                                                                       cmap="rainbow"), marker_size=400,
                                    marker_vectors=pol_vecfield, marker_vectors_length=100, marker_vector_width=2,
                                    vec_edge_width=0.5, vec_length=20,
                                    marker_vectors_color=visuals.color_scalar(m_charge[charge_pol_linked_idxs],
                                                                              manual_vminmax=[-1, 1],
                                                                              cmap="rainbow"), )

### -- Load Defect(s) Polarisation(s) --

In [ ]:
patch_polarisation = ["radius", 40]
# patch_polarisation = ["nearest", 80]
patch_type = patch_polarisation[0]
patch_size = patch_polarisation[1]

if patch_type == "radius":
    patch_label = f"r-{patch_size}{img_unit}"
elif patch_type == "nearest":
    patch_label = f"k-{patch_size}"
else:
    print(f"[!] Unknown patch type: {patch_type}")

pol_vecfield = datahandler.load_array(name=f"def-pol_2dcurved_{patch_label}", folderpath=resdata_dir_layer)
charge_pol_linked_idxs = datahandler.load_array(name=f"def-pol_2dcurved_{patch_label}_idxs",
                                                folderpath=resdata_dir_layer).astype(int)

In [ ]:
rotation_angles = [1, 1, 1]
sph_proj_phi, sph_proj_theta = analysis.spherical_project(pts=layer_mesh.vertices, rotate=rotation_angles)
vec_dir_phi, vec_dir_theta = analysis.spherical_project_vectors(directors_2dcurved_avg[:, :3],
                                                                directors_2dcurved_avg[:, 3:], rotate=rotation_angles)
pol_dir_phi, pol_dir_theta = analysis.spherical_project_vectors(pol_vecfield[:, :3], pol_vecfield[:, 3:],
                                                                rotate=rotation_angles)
visuals.plot_spherical_projection(
    phi=sph_proj_phi,
    theta=sph_proj_theta,
    intensities=proj_layer,
    vec_pos_phi=sph_proj_phi[idxs_sel],
    vec_pos_theta=sph_proj_theta[idxs_sel],
    vec_dir_phi=vec_dir_phi,
    vec_dir_theta=vec_dir_theta,
    vec_cmap_label="order scalar $S$",
    hexgridsize=400,
    scale_factor=2,
    cmap="Greys_r", vec_width=0.0015,
    veccolor=S_2dcurv, vec_manual_vminmax=[0, 1],
    arrow_alpha=1.0, figsize=(22, 8),
    marker_idxs=idxs_sel[defect_idxs_calc],
    marker_color=visuals.color_scalar(m_charge, manual_vminmax=[-1, 1], cmap="rainbow"),
    marker_vec=(pol_dir_phi, pol_dir_theta, idxs_sel[pol_idxs]),
    marker_vec_scale=20, marker_vec_width=0.005, aspect="equal",
    marker_vec_color=visuals.color_scalar(m_charge[charge_pol_linked_idxs], manual_vminmax=[-1, 1],
                                          cmap="rainbow"),
    savefig=os.path.join(resfig_dir_layer, f"defect-polarisations.png")
)

In [ ]:
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg, mesh_shading="flat",
                                    mesh_vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                          cmap="Greys_r"),
                                    vec_colors="red", vec_edge_width=0.08, vec_length=10)

In [ ]:
reload(visuals)
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg, mesh_shading="flat",
                                    mesh_vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                          cmap="Greys_r"),
                                    vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]),
                                    markers=layer_mesh.vertices[idxs_sel[defect_idxs_calc]], vec_edge_width=0.3,
                                    marker_colors=visuals.color_scalar(m_charge, manual_vminmax=[-1, 1],
                                                                       cmap="rainbow"), marker_size=400,
                                    vec_length=10)

In [ ]:
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg, mesh_shading="flat",
                                    mesh_vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                          cmap="Greys_r"),
                                    vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]),
                                    markers=layer_mesh.vertices[idxs_sel[defect_idxs_calc]], vec_edge_width=0.3,
                                    marker_colors=visuals.color_scalar(m_charge, manual_vminmax=[-1, 1],
                                                                       cmap="rainbow"), marker_size=400,
                                    marker_vectors=pol_vecfield, marker_vectors_length=20, vec_length=10,
                                    marker_vector_width=3,
                                    marker_vectors_color=visuals.color_scalar(m_charge[charge_pol_linked_idxs],
                                                                              manual_vminmax=[-1, 1],
                                                                              cmap="rainbow"), )

### |5| Criss-Cross

In [ ]:
found_analysed_layers = [
    (layer, f.split("_")[-1].split(".csv")[0])
    for layer in layer_names
    for f in os.listdir(os.path.join(resdata_dir, layer))
    if f.startswith("S-order-init_2dcurved") and f.endswith(".csv")
]

print(f"Found layer(s) and patch label(s): {found_analysed_layers}")

(layer_name_1, patch_label_1), (layer_name_2, patch_label_2) = [found_analysed_layers[0], found_analysed_layers[-1]]
print(
    f"Choosing layer 1 with name {layer_name_1} and patch label {patch_label_1} | layer 2 with name {layer_name_2} and patch label {patch_label_2}")

In [ ]:
try:
    layer_mesh_1 = datahandler.load_mesh(os.path.join(resdata_dir, layer_name_1, "layer_mesh.ply"),
                                         recalc_normals=False, clean=False)
except:
    print("Could not find layer mesh, using smampling mesh instead..")
    layer_mesh_1 = datahandler.load_mesh(os.path.join(resdata_dir, "sampling_mesh.ply"), recalc_normals=True,
                                         clean=False)

try:
    layer_mesh_2 = datahandler.load_mesh(os.path.join(resdata_dir, layer_name_2, "layer_mesh.ply"),
                                         recalc_normals=False, clean=False)
except:
    print("Could not find layer mesh, using smampling mesh instead..")
    layer_mesh_2 = datahandler.load_mesh(os.path.join(resdata_dir, "sampling_mesh.ply"), recalc_normals=True,
                                         clean=False)

In [ ]:
# layer_name_1 = 'proj_4.9_to_5.1_um'
# patch_label_1 = 'r-30um'
# layer_name_2 = 'proj_8.9_to_9.1_um'
# patch_label_2 = 'r-30um'
# (layer_name_1, patch_label_1), (layer_name_2, patch_label_2) = found_analysed_layers[:2]
crisscross_mag, field_1, field_2 = analysis.layers_crisscross(layer_name_1=layer_name_1, patch_label_1=patch_label_1,
                                                              layer_name_2=layer_name_2, patch_label_2=patch_label_2,
                                                              resdata_dir=resdata_dir,
                                                              director_name_prefix="directors-avg-init_2dcurved_")
datahandler.save_array(crisscross_mag,
                       f"{layer_name_1}_{patch_label_1}_VS_{layer_name_2}_{patch_label_2}_crisscross_mag",
                       header="strength", folderpath=resdata_dir)

visuals.plot_hist(crisscross_mag, title="Criss-Cross Strength", xlim=[0, 1],
                  savefig=os.path.join(resfig_dir,
                                       f"{layer_name_1}_{patch_label_1}_VS_{layer_name_2}_{patch_label_2}_hist_crisscross_mag.png"))

visuals.plot_dir_field(directors=field_1, veclength=10, view_init=(20, 0),
                       veccolor=crisscross_mag,
                       cmap_label="Criss-Cross Strength", show_axes=False, manual_vminmax=[0, 1],
                       cmap="coolwarm",
                       savefig=os.path.join(resfig_dir,
                                            f"{layer_name_1}_{patch_label_1}_VS_{layer_name_2}_{patch_label_2}_nematic-field_crisscross_mag.png"))
rotation_angles = [1, 1, 1]
plotted_vecfields = np.concatenate((field_2, field_1), axis=0)
sph_proj_phi, sph_proj_theta = analysis.spherical_project(pts=plotted_vecfields[:, :3],
                                                          rotate=rotation_angles)
vec_dir_phi, vec_dir_theta = analysis.spherical_project_vectors(plotted_vecfields[:, :3],
                                                                plotted_vecfields[:, 3:],
                                                                rotate=rotation_angles)
visuals.plot_spherical_projection(
    phi=sph_proj_phi,
    theta=sph_proj_theta,
    intensities=np.ones_like(sph_proj_phi),
    vec_pos_phi=sph_proj_phi,
    vec_pos_theta=sph_proj_theta,
    vec_dir_phi=vec_dir_phi,
    vec_dir_theta=vec_dir_theta,
    vec_cmap_label="criss cross magnitude",
    hexgridsize=200,
    scale_factor=5, alpha=0.0,
    cmap="Greys_r", vec_width=0.0015,
    veccolor=np.concatenate((np.ones(len(field_2)) * 0.5, crisscross_mag), axis=0),
    vec_manual_vminmax=[0, 1],
    arrow_alpha=0.8, figsize=(22, 8), vec_cmap="coolwarm",
    aspect="equal",
    savefig=os.path.join(resfig_dir,
                         f"{layer_name_1}_{patch_label_1}_VS_{layer_name_2}_{patch_label_2}_nematic-field_crisscross_mag.png")
)


In [ ]:
mask = (plotted_vecfields[:, :3] - np.mean(plotted_vecfields[:, :3], axis=0))[:, 2] > 0
mask = np.ones(plotted_vecfields.shape[0], dtype=bool)
visuals.view_colored_mesh_dir_field(mesh=layer_mesh_1, directors=plotted_vecfields[mask],
                                    vec_colors=visuals.color_scalar(
                                        np.concatenate((np.ones(len(field_2)) * 0.5, crisscross_mag), axis=0)[mask],
                                        manual_vminmax=[0, 1], cmap="coolwarm", ),
                                    mesh_vert_colors="black",
                                    vec_edge_width=0.5, vec_length=10)

In [ ]:
mask = (plotted_vecfields[:, :3] - np.mean(plotted_vecfields[:, :3], axis=0))[:, 2] > 0
mask = np.ones(plotted_vecfields.shape[0], dtype=bool)
visuals.view_colored_mesh_dir_field(mesh=layer_mesh_1, directors=plotted_vecfields[mask],
                                    vec_colors=visuals.color_scalar(
                                        np.concatenate((np.ones(len(field_2)) * 0.5, crisscross_mag), axis=0)[mask],
                                        manual_vminmax=[0, 1], cmap="coolwarm", ),
                                    mesh_vert_colors="black",
                                    vec_edge_width=0.5, vec_length=10, img=img_raw, scale=img_scale)

In [ ]:
proj_layer_1 = datahandler.load_array("intensities", folderpath=os.path.join(resdata_dir, layer_name_1))
proj_layer_2 = datahandler.load_array("intensities", folderpath=os.path.join(resdata_dir, layer_name_2))
visuals.view_colored_mesh_multiple(mesh_list=[layer_mesh_1, layer_mesh_2],
                                   vert_colors_list=[visuals.color_scalar(analysis.normalise_range(proj_layer_1),
                                                                          cmap="Greens_r"),
                                                     visuals.color_scalar(analysis.normalise_range(proj_layer_2),
                                                                          cmap="Blues_r")])

In [ ]:
S_2dcurv_layer_1 = datahandler.load_array(name=f"S-order-init_2dcurved_{patch_label_1}",
                                          folderpath=os.path.join(resdata_dir, layer_name_1))
S_2dcurv_layer_2 = datahandler.load_array(name=f"S-order-init_2dcurved_{patch_label_2}",
                                          folderpath=os.path.join(resdata_dir, layer_name_2))
field_1_raw = datahandler.load_array(f"directors_2dcurved",
                                     folderpath=os.path.join(resdata_dir, layer_name_1))
field_2_raw = datahandler.load_array(f"directors_2dcurved",
                                     folderpath=os.path.join(resdata_dir, layer_name_2))

In [ ]:
visuals.view_colored_mesh_dir_field(mesh=trimesh.util.concatenate(layer_mesh_1, layer_mesh_2),
                                    directors=plotted_vecfields,
                                    vec_colors=visuals.color_scalar(
                                        np.concatenate((S_2dcurv_layer_2, S_2dcurv_layer_1)), manual_vminmax=[0, 1],
                                        cmap="Spectral"),
                                    mesh_vert_colors=visuals.color_scalar(np.concatenate(
                                        (analysis.normalise_range(proj_layer_1),
                                         analysis.normalise_range(proj_layer_2))),
                                        cmap="Greys_r"),
                                    vec_edge_width=0.3, vec_length=20)

# (experimental Oriol Gastruloids)

## -- Load Libraries --

In [ ]:
from scripts import extra_gastruloids
from batch_analysis.automated_scripts import run_projection, run_tan_orient_extract, run_cylindrical_analysis, \
    run_sphi_decomposition

for module in (run_projection, run_tan_orient_extract, run_cylindrical_analysis, extra_gastruloids,
               run_sphi_decomposition):
    reload(module)

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.interpolate import interp1d
import matplotlib.cm as cm
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.ndimage import gaussian_filter1d

## Manual run

In [ ]:
gastr_mesh = datahandler.load_mesh(os.path.join(resdata_dir, "sampling_mesh.ply"), recalc_normals=False, clean=False)

In [ ]:
# img_path = '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/96/200/Gas4.tif'
centerline_fitted, mesh_s, mesh_rho, mesh_phi = run_cylindrical_analysis.main(
    img_path=img_path, overwrite=False, show_figures=False, render=False)
visuals.view_colored_mesh(gastr_mesh, vert_colors="white", markers=centerline, mesh_shading="flat",
                          mesh_opacity=0.6, mesh_blending="translucent_no_depth",
                          marker_colors=visuals.color_scalar(np.linspace(0, 1, len(centerline)), cmap="Blues"))
# # ==== 3D Render Curvi-linear Coordinates ====
visuals.view_colored_mesh_multiple([gastr_mesh, gastr_mesh, gastr_mesh, gastr_mesh],
                                   vert_colors_list=["white",
                                                     visuals.color_scalar(mesh_s, manual_vminmax=[0, mesh_s.max()],
                                                                          cmap="inferno"),
                                                     visuals.color_scalar(mesh_rho, manual_vminmax=[0, mesh_rho.max()],
                                                                          cmap="Spectral"),
                                                     visuals.color_scalar(mesh_phi, manual_vminmax=[-np.pi, np.pi],
                                                                          cmap="hsv")],
                                   markers=centerline_fitted, mesh_shading="flat",
                                   mesh_opacity_list=[0.6, 1.0, 1.0, 1.0],
                                   marker_colors=visuals.color_scalar(np.linspace(0, 1, len(centerline_fitted)),
                                                                      cmap="Blues"))

### -- Load Layer --

In [ ]:
found_projected_layers = [
    d for d in os.listdir(resdata_dir)
    if os.path.isdir(os.path.join(resdata_dir, d))
]

found_analysed_layers = [
    (layer, f.split("_")[-1].split(".csv")[0])
    for layer in found_projected_layers
    for f in os.listdir(os.path.join(resdata_dir, layer))
    if f.startswith("S-order-init_2dcurved") and f.endswith(".csv")
]
print(found_projected_layers, "\n", found_analysed_layers)

In [ ]:
layer_label = 'proj_14.95_to_15.05_um'
patch_avg = ["radius", 30]
# ==== Load Projected Result ====
resdata_dir_layer = os.path.join(resdata_dir, layer_label)
resfig_dir_layer = os.path.join(resfig_dir, layer_label)
proj_layer = datahandler.load_array("intensities", folderpath=resdata_dir_layer)
proj_layer /= proj_layer.max()

# ==== Load 2D+ Directors ====
idxs_sel = datahandler.load_array("calcindeces", folderpath=resdata_dir_layer).astype(int)
tan_x = datahandler.load_array("tan_x", folderpath=resdata_dir_layer)
tan_y = datahandler.load_array("tan_y", folderpath=resdata_dir_layer)
directors_2dcurved = datahandler.load_array("directors_2dcurved", folderpath=resdata_dir_layer)

# ==== Load 2D+ nematic order ====
patch_type = patch_avg[0]
patch_size = patch_avg[1]

if patch_type == "radius":
    patch_label = f"r-{patch_size}{img_unit}"
    title_hist = f"Avg over {patch_size}{img_unit}: Order Scalar $S$"
    title_render = f"Avg over {patch_size}{img_unit}: Average Directors"
elif patch_type == "nearest":
    patch_label = f"k-{patch_size}"
    title_hist = f"Avg over {patch_size - 1} neighbours: Order Scalar $S$"
    title_render = f"Avg over {patch_size - 1} neighbours: Average Directors"
else:
    neigh_idxs = patch_label = title_hist = title_render = None
    print(f"[!] Unknown patch type: {patch_type}")
directors_2dcurved_avg = datahandler.load_array(f"directors-avg-init_2dcurved_{patch_label}",
                                                folderpath=resdata_dir_layer)
S_2dcurv = datahandler.load_array(f"S-order-init_2dcurved_{patch_label}", folderpath=resdata_dir_layer)

In [ ]:
# ==== 3D Render Results ====
visuals.view_colored_mesh_dir_field(mesh=gastr_mesh, directors=directors_2dcurved,
                                    vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]),
                                    mesh_vert_colors="white")

### |2| Project Layer on 3D Curve

In [ ]:
phi_new_zero = extra_gastruloids.find_phi_intensity_max(phi_vals=mesh_phi, intensity_vals=proj_layer)
phi_new_zero -= 0.6
mesh_phi = extra_gastruloids.shift_angle_periodic(angle=mesh_phi, angle_zerobase=phi_new_zero)
dir_s, dir_rho, dir_phi = extra_gastruloids.cylindrical_along_curve(points=directors_2dcurved[:, :3],
                                                                    curve=centerline_fitted)
dir_phi = extra_gastruloids.shift_angle_periodic(angle=dir_phi, angle_zerobase=phi_new_zero)

visuals.plot_cylindrical_projection(phi=mesh_phi, rho=mesh_rho, s=mesh_s, colors=proj_layer, aspect="equal",
                                    title=f"Projected Intensities \n{layer_label}", hexsize=300, cmap="inferno")

low_cutoff_phi = -np.pi / 3
high_cutoff_phi = np.pi / 3

mesh_s_cropped = extra_gastruloids.crop_by_angles(values=mesh_s, angles=mesh_phi, angle_low_cutoff=low_cutoff_phi,
                                                  angle_high_cutoff=high_cutoff_phi)
mesh_rho_cropped = extra_gastruloids.crop_by_angles(values=mesh_rho, angles=mesh_phi, angle_low_cutoff=low_cutoff_phi,
                                                    angle_high_cutoff=high_cutoff_phi)
proj_layer_cropped = extra_gastruloids.crop_by_angles(values=proj_layer, angles=mesh_phi,
                                                      angle_low_cutoff=low_cutoff_phi,
                                                      angle_high_cutoff=high_cutoff_phi)
mesh_phi_cropped = extra_gastruloids.crop_by_angles(values=mesh_phi, angles=mesh_phi, angle_low_cutoff=low_cutoff_phi,
                                                    angle_high_cutoff=high_cutoff_phi)

dir_s_cropped = extra_gastruloids.crop_by_angles(values=dir_s, angles=dir_phi, angle_low_cutoff=low_cutoff_phi,
                                                 angle_high_cutoff=high_cutoff_phi)
dir_rho_cropped = extra_gastruloids.crop_by_angles(values=dir_rho, angles=dir_phi, angle_low_cutoff=low_cutoff_phi,
                                                   angle_high_cutoff=high_cutoff_phi)
dir_phi_cropped = extra_gastruloids.crop_by_angles(values=dir_phi, angles=dir_phi, angle_low_cutoff=low_cutoff_phi,
                                                   angle_high_cutoff=high_cutoff_phi)
visuals.plot_scatter(x=mesh_phi_cropped, y=proj_layer_cropped, title="Projection Intensity vs. Angle",
                     xlabel=r"$\phi$ (rad)", vert_line=[low_cutoff_phi, high_cutoff_phi],
                     ylabel="Projection Intensity (a.u.)", xlim=[-np.pi, np.pi])
visuals.plot_cylindrical_projection(phi=mesh_phi_cropped, rho=mesh_rho_cropped, s=mesh_s_cropped,
                                    colors=proj_layer_cropped, aspect="equal",
                                    title=f"Projected Intensities \n{layer_label}", hexsize=400, figsize=(10, 5),
                                    cmap="inferno")
visuals.plot_rho_profile(mesh_s=mesh_s_cropped, mesh_rho=mesh_rho_cropped, mesh_phi=mesh_phi_cropped, img_unit=img_unit)



### |3| Decompose Nematic Field

In [ ]:
e_s, e_phi, e_rho = extra_gastruloids.create_s_phi_basis(points=directors_2dcurved[:, :3], curve=centerline_fitted,
                                                         normals=gastr_mesh.vertex_normals[idxs_sel])
e_s_cropped = extra_gastruloids.crop_by_angles(values=e_s, angles=dir_phi, angle_low_cutoff=low_cutoff_phi,
                                               angle_high_cutoff=high_cutoff_phi)
e_phi_cropped = extra_gastruloids.crop_by_angles(values=e_phi, angles=dir_phi, angle_low_cutoff=low_cutoff_phi,
                                                 angle_high_cutoff=high_cutoff_phi)
e_rho_cropped = extra_gastruloids.crop_by_angles(values=e_rho, angles=dir_phi, angle_low_cutoff=low_cutoff_phi,
                                                 angle_high_cutoff=high_cutoff_phi)
directors_2dcurved_cropped = extra_gastruloids.crop_by_angles(values=directors_2dcurved, angles=dir_phi,
                                                              angle_low_cutoff=low_cutoff_phi,
                                                              angle_high_cutoff=high_cutoff_phi)

In [ ]:
# e_s_mesh, e_phi_mesh, e_rho_mesh = extra_gastruloids.create_s_phi_basis(points=gastr_mesh.vertices,
#                                                                         curve=centerline_fitted,
#                                                                         normals=gastr_mesh.vertex_normals)
# freq = 50
# visuals.view_3d_vector_field_multiple(
#     vec_pos=[gastr_mesh.vertices[::freq], gastr_mesh.vertices[::freq]],
#     vec_dir=[e_s_mesh[::freq], e_phi_mesh[::freq]], vec_colors=["orange", "lightblue"], vec_names=["e_s", "e_phi"])

In [ ]:
# directors_2dcurved_cropped[:, 3:] = e_s_cropped + e_phi_cropped
# directors_2dcurved_cropped[:, 3:] /= np.linalg.norm(directors_2dcurved_cropped[:, 3:])
# visuals.view_3d_vector_field_multiple(
#     vec_pos=[directors_2dcurved_cropped[:, :3], directors_2dcurved_cropped[:, :3], directors_2dcurved_cropped[:, :3]],
#     vec_dir=[directors_2dcurved_cropped[:, 3:], e_s, e_phi], vec_colors=["white", "orange", "purple"],
#     vec_names=["dir", "e_s", "e_phi"])
# visuals.view_3d_vector_field_multiple(
#     vec_pos=[directors_2dcurved_cropped[:, :3], directors_2dcurved_cropped[:, :3]],
#     vec_dir=[e_s, e_phi], vec_colors=["orange", "lightblue"], vec_names=["e_s", "e_phi"])
# visuals.view_colored_mesh(gastr_mesh, vert_colors=visuals.color_scalar(
#     analysis.interpolate_on_mesh(values=S_2dcurv, mesh=gastr_mesh, value_idxs=idxs_sel), manual_vminmax=[0, 1],
#     cmap="Spectral"), mesh_shading="flat", mesh_blending="opaque", mesh_opacity=1.0)

In [ ]:
# ==== Tune Curved Nematic Analysis Number of Neighbours ====
patch_avg = ["radius", 50]
# patch_avg = ["nearest", 20]
patch_type = patch_avg[0]
patch_size = patch_avg[1]

# ==== Tune Plotting parameters ====
vec_length = 20
plot2d_view = (20, 0)
histfigsize = (4, 3)
renderfigsize = (6, 5)
veccoords = directors_2dcurved_cropped[:, :3]
if patch_type == "radius":
    neigh_idxs = analysis.coord_search_radius(veccoords, r=patch_size)
    patch_label = f"r-{patch_size}{img_unit}"
    title_hist = f"Avg over {patch_size}{img_unit}: Order Scalar $S$"
    title_render = f"Avg over {patch_size}{img_unit}: Average Directors"
elif patch_type == "nearest":
    neigh_idxs = analysis.coord_search_neighbours(veccoords, k=patch_size, n_process=8)
    patch_label = f"k-{patch_size}"
    title_hist = f"Avg over {patch_size - 1} neighbours: Order Scalar $S$"
    title_render = f"Avg over {patch_size - 1} neighbours: Average Directors"
else:
    neigh_idxs = patch_label = title_hist = title_render = None
    print(f"[!] Unknown patch type: {patch_type}")

# ==== Calculate Curved Nematic Order ====
S_2dcurv_sphi_cropped, n_avg_2dcurv_sphi_cropped, q_sphi_cropped = analysis.avg_tan_nem_tens(t1_cov=e_s_cropped,
                                                                                             t2_cov=e_phi_cropped,
                                                                                             directors=directors_2dcurved_cropped,
                                                                                             neigh_idxs=neigh_idxs,
                                                                                             return_qij_bar=True)

# ==== Save Curved Nematic Order ====
# datahandler.save_array(S_2dcurv, name=f"S-order_2dcurved_{patch_label}", header="S", folderpath=resdata_dir_layer)
# datahandler.save_array(np.column_stack((veccoords, n_avg_2dcurv)), name=f"directors-avg_2dcurved_{patch_label}",
#                        header="x,y,z,vx,vy,vz", folderpath=resdata_dir_layer)

# ==== Plot Curved Nematic Order ====
directors_2dcurved_cropped_avg_sphi = directors_2dcurved_cropped.copy()
directors_2dcurved_cropped_avg_sphi[:, 3:] = n_avg_2dcurv_sphi_cropped
# savefig_render = os.path.join(resfig_dir_layer, f"field_avg-nematic_{patch_label}.png")
# savefig_hist = os.path.join(resfig_dir_layer, f"hist_order-s_{patch_label}.png")
savefig_render = ""
savefig_hist = ""

visuals.plot_dir_field(directors=directors_2dcurved_cropped_avg_sphi, veclength=vec_length, view_init=plot2d_view,
                       veccolor=S_2dcurv_sphi_cropped, cmap_label="order scalar $S$", title=title_render,
                       manual_vminmax=[0, 1],
                       savefig=savefig_render, figsize=renderfigsize, show_axes=False)
visuals.plot_hist(array=S_2dcurv_sphi_cropped, title=title_hist, savefig=savefig_hist, figsize=histfigsize, xlim=[0, 1])

s_bin_centers_cropped, Q_ss_cropped, Q_ss_mean_cropped, Q_phiphi_cropped, Q_phiphi_mean_cropped, Q_sphi_cropped, Q_sphi_mean_cropped = extra_gastruloids.decompose_q_sphi(
    q_sphi=q_sphi_cropped, s_coords=dir_s_cropped, num_bins=50)
visuals.plot_qsphi_profiles(dir_s=dir_s_cropped, s_bin_centers=s_bin_centers_cropped, Q_ss=Q_ss_cropped,
                            Q_ss_mean=Q_ss_mean_cropped, Q_phiphi=Q_phiphi_cropped,
                            Q_phiphi_mean=Q_phiphi_mean_cropped, Q_sphi=Q_sphi_cropped, Q_sphi_mean=Q_sphi_mean_cropped,
                            y_limits=[-0.5, 0.5])

visuals.plot_qsphi_profiles_separated_phi(dir_s=dir_s_cropped, dir_phi=dir_phi_cropped,
                                          s_bin_centers=s_bin_centers_cropped, Q_ss=Q_ss_cropped,
                                          Q_ss_mean=Q_ss_mean_cropped, Q_phiphi=Q_phiphi_cropped,
                                          Q_phiphi_mean=Q_phiphi_mean_cropped, Q_sphi=Q_sphi_cropped,
                                          Q_sphi_mean=Q_sphi_mean_cropped,
                                          y_limits=[-0.5, 0.5])


In [ ]:
visuals.plot_qsphi_profiles(dir_s=dir_s_cropped, s_bin_centers=s_bin_centers_cropped, Q_ss=Q_ss_cropped,
                            Q_ss_mean=Q_ss_mean_cropped, Q_phiphi=Q_phiphi_cropped,
                            Q_phiphi_mean=Q_phiphi_mean_cropped, Q_sphi=Q_sphi_cropped, Q_sphi_mean=Q_sphi_mean_cropped,
                            y_limits=[-0.5, 0.5])

In [ ]:
# ==== 3D Render Curved Nematic Order ====
vec_length = 10
vec_edge_width = 1
visuals.view_colored_mesh_dir_field(mesh=gastr_mesh, directors=directors_2dcurved_cropped,
                                    vec_colors=visuals.color_scalar(S_2dcurv_sphi_cropped, manual_vminmax=[0, 1]),
                                    mesh_vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                          cmap="Greys_r"),
                                    vec_length=vec_length, vec_edge_width=vec_edge_width)

In [ ]:
# ==== 3D Render Curved Nematic Order ====
vec_length = 10
vec_edge_width = 1
visuals.view_colored_mesh_dir_field(mesh=gastr_mesh, directors=directors_2dcurved_cropped,
                                    vec_colors=visuals.color_scalar(S_2dcurv_sphi_cropped, manual_vminmax=[0, 1]),
                                    mesh_vert_colors=visuals.color_scalar(mesh_phi, manual_vminmax=[-np.pi, np.pi],
                                                                          cmap="hsv"),
                                    vec_length=vec_length, vec_edge_width=vec_edge_width)

### -- Plot decompositions --

In [ ]:
path_all = [
    "/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/72h_300_Gas1.tif",
    "/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/96h_300_Gas3.tif",
    "/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/104h_300_Gas1.tif",
    "/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/112h_300_Gas2.tif",
    "/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/120h_300_Gas9.tif"]
projection_label = "proj_-36.0_to_-35.0_um"  #"proj_-21.0_to_-20.0_um"
time_points_all = []
all_packets = []
reload(extra_gastruloids)
for path in path_all:
    decomposition_packet = extra_gastruloids.proj_nem_on_sphi(img_path=path, layer_label=projection_label)
    all_packets.append(decomposition_packet)
    time_points_all.append(int(path.split("h_")[0].split("/")[-1]))
s_bin_centers_all = []
Q_ss_all = []
Q_phiphi_all = []
Q_sphi_all = []
Q_ss_mean_all = []
Q_phiphi_mean_all = []
Q_sphi_mean_all = []
dir_s_all = []
dir_phi_all = []
dir_rho_all = []
mesh_s_all = []
mesh_phi_all = []
mesh_rho_all = []
for i, packet in enumerate(all_packets):
    mesh_coords, dir_coords, q_decomposition = packet
    s_bin_centers, Q_ss, Q_ss_mean, Q_phiphi, Q_phiphi_mean, Q_sphi, Q_sphi_mean = q_decomposition
    dir_s, dir_rho, dir_phi = dir_coords
    mesh_s, mesh_rho, mesh_phi = mesh_coords
    s_bin_centers_all.append(s_bin_centers)
    Q_ss_all.append(Q_ss)
    Q_phiphi_all.append(Q_phiphi)
    Q_sphi_all.append(Q_sphi)
    Q_ss_mean_all.append(Q_ss_mean)
    Q_phiphi_mean_all.append(Q_phiphi_mean)
    Q_sphi_mean_all.append(Q_sphi_mean)
    dir_s_all.append(dir_s)
    dir_phi_all.append(dir_phi)
    dir_rho_all.append(dir_rho)
    mesh_s_all.append(mesh_s)
    mesh_phi_all.append(mesh_phi)
    mesh_rho_all.append(mesh_rho)

mesh_s_all_min = [np.min(s) for s in mesh_s_all]
mesh_s_all_max = [np.max(s) for s in mesh_s_all]
mesh_rho_all_min = [np.min(rho) for rho in mesh_rho_all]
mesh_rho_all_max = [np.max(rho) for rho in mesh_rho_all]
mesh_s_all_mean = [np.mean(s) for s in mesh_s_all]
mesh_rho_all_mean = [np.mean(rho) for rho in mesh_rho_all]

In [ ]:
plt.figure()
plt.plot(time_points_all, sphericity, "o-", c="g")
plt.title(r"Sphericity Over Time: $\Psi = \frac{\pi^{1/3}(6V)^{2/3}}{A}$")
plt.xlabel("Time (h)")
plt.yticks(np.arange(0, 1.1, 0.1))
plt.grid()
plt.ylim(0, 1)
plt.show()
plt.figure()
plt.plot(time_points_all, [s.max() / (2 * rho.max()) for s, rho in zip(mesh_s_all, mesh_rho_all)], "o-", c="g")
plt.title("A<->P body length / maximum diameter")
plt.xlabel("Time (h)")
plt.show()
plt.figure()
plt.plot(time_points_all, mesh_s_all_max, "o-", c="r", label="max arc length")
plt.plot(time_points_all, mesh_rho_all_max, "o-", c="b", label="max diameter")
plt.xlabel("Time (h)")
plt.ylabel(r"Length ($\mu$m)")
plt.legend()
plt.show()
plt.figure()
plt.plot(time_points_all, mesh_s_all_mean, "o-", c="r", label="mean arc length")
plt.plot(time_points_all, mesh_rho_all_mean, "o-", c="b", label="mean diameter")
plt.xlabel("Time (h)")
plt.ylabel(r"Length ($\mu$m)")
plt.yscale("log")
plt.legend()
plt.show()

In [ ]:
img_unit = "um"
normalise_bodyaxis = False
cbarlabel = "Developmental time (h)"
ylabel = r"$\rho$" + f" ({img_unit})"
title = r"Thickness $\rho$ Profile" + f" ({img_unit})"
if normalise_bodyaxis:
    mesh_s_norm_all = []
    for i in range(len(mesh_s_all)):
        s_min, s_max = mesh_s_all[i].min(), mesh_s_all[i].max()
        s_norm = (mesh_s_all[i] - s_min) / (s_max - s_min)
        mesh_s_norm_all.append(s_norm)
    savefig = f"/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/{projection_label}_thickness_profile_normalised-body-axis.png"
    xlabel = "Normalised arc length $s/L$"
    visuals.plot_rho_profile_evolution(time_points_all=time_points_all, mesh_s_all=mesh_s_norm_all,
                                       mesh_rho_all=mesh_rho_all, xlabel=xlabel, title=title, ylabel=ylabel,
                                       cbarlabel=cbarlabel, savefig=savefig)
else:
    savefig = f"/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/{projection_label}_thickness_profile.png"
    xlabel = r"Arc length $s$" + f" ({img_unit})"
    visuals.plot_rho_profile_evolution(time_points_all=time_points_all, mesh_s_all=mesh_s_all,
                                       mesh_rho_all=mesh_rho_all, xlabel=xlabel, title=title, ylabel=ylabel,
                                       cbarlabel=cbarlabel, savefig=savefig)

In [ ]:
img_unit = "um"
normalise_bodyaxis = True
cbarlabel = "Developmental time (h)"
title = r"Nematic order $Q_{s\phi}$ Profile" + f" \n {projection_label}"
q_mean_all_dict = {
    "Q_ss": Q_ss_mean_all,
    "Q_phiphi": Q_phiphi_mean_all,
    "Q_sphi": Q_sphi_mean_all,
}
if normalise_bodyaxis:
    s_bin_centers_norm_all = []
    for i in range(len(dir_s_all)):
        s_min, s_max = dir_s_all[i].min(), dir_s_all[i].max()
        s_bin_norm = (s_bin_centers_all[i] - s_min) / (s_max - s_min)
        s_bin_centers_norm_all.append(s_bin_norm)
    savefig = f"/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/{projection_label}_q-s-phi_profile_normalised-body-axis.png"
    xlabel = "Normalised arc length $s/L$"
    visuals.plot_qsphi_profile_evolution(time_points_all=time_points_all, s_bin_centers_all=s_bin_centers_norm_all,
                                         q_mean_all_dict=q_mean_all_dict,
                                         xlabel=xlabel, title=title, cbarlabel=cbarlabel, savefig=savefig,
                                         ylim=[-0.5, 0.5])
else:
    savefig = f"/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/{projection_label}_q-s-phi_profile.png"
    xlabel = r"Arc length $s$" + f" ({img_unit})"
    visuals.plot_qsphi_profile_evolution(time_points_all=time_points_all, s_bin_centers_all=s_bin_centers_all,
                                         q_mean_all_dict=q_mean_all_dict,
                                         xlabel=xlabel, title=title, cbarlabel=cbarlabel, savefig=savefig,
                                         ylim=[-0.5, 0.5])

## Batch Setup

In [ ]:


img_path_list_exp4_cleaned = sorted(glob.glob(
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/**/*.tif',
    recursive=True))
print(f"Found {len(img_path_list_exp4_cleaned)} images !")

In [ ]:
min_dist_all = 15 - 0.05
max_dist_all = 15 + 0.05
num_dist_all = 30
img_unit = "um"
layer_label_all = f"proj_{min_dist_all}_to_{max_dist_all}_{img_unit}"
proj_mode_all = "mean"
dir_extr_patch_size_all = 30
low_cutoff_phi_all = -np.pi / 3
high_cutoff_phi_all = np.pi / 3
q_decomp_radius_all = 50.0

In [ ]:
gastruloid_batch_output_folder = '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/URI_25022026_PR_NEMO/EXP4_filter_membrane/!batch-analysis'
dataset_nematic_binned_name = f"df_nematic_binned_{layer_label_all}_r={q_decomp_radius_all}.pkl"
dataset_nematic_profile_name = f"df_nematic_raw_{layer_label_all}_r={q_decomp_radius_all}.pkl"
dataset_morphology_name = "df_morpho.pkl"
q_decomp_label_all = f"r-{q_decomp_radius_all}um"

In [ ]:
img_path_list_exp4_cleaned_only72h = [p for p in img_path_list_exp4_cleaned if p.split(os.sep)[-3] == '72']

## Batch Run

In [ ]:
###################
# Project a layer #
###################
for i, path in enumerate(img_path_list_exp4_cleaned):
    print(f"=========== [Progress {100 * np.round(i / len(img_path_list_exp4_cleaned), 2)}%] {path} ===========")
    run_projection.main(img_path=path, flip_normals=True, dist_min=min_dist_all, dist_max=max_dist_all,
                        dist_num=num_dist_all,
                        proj_mode="mean", render=False, show_plots=False)

In [ ]:
####################################
# Extract tangential nematic field #
####################################
for i, path in enumerate(img_path_list_exp4_cleaned):
    print(f"=========== [Progress {100 * np.round(i / len(img_path_list_exp4_cleaned), 2)}%] {path} ===========")
    run_tan_orient_extract.main(img_path=path, layer_label=layer_label_all, render=False, patch_mode="radius",
                                patch_size=dir_extr_patch_size_all, normal_validity_k=20, normal_validity_thresh=0.99,
                                compute_num=3000, grid_n_2dcurve_analysis=30, debug_2dcurve_analysis=False,
                                show_figures=False)

In [ ]:
###################
# Find body axes  #
###################
for i, path in enumerate(img_path_list_exp4_cleaned):
    print(f"=========== [Progress {100 * np.round(i / len(img_path_list_exp4_cleaned), 2)}%] {path} ===========")
    timepoint = int(path.split("_CLEAN/")[1].split("/")[0])
    run_cylindrical_analysis.main(img_path=path, overwrite=False, show_figures=False,
                                  voxel_size=2 if timepoint == 72 else 3,
                                  spline_smooth_factor=400,
                                  render=False, force_pca_use=True if timepoint == 72 else False)

In [ ]:
###################
# Decompose layer #
###################
for i, path in enumerate(img_path_list_exp4_cleaned):
    print(f"=========== [Progress {100 * np.round(i / len(img_path_list_exp4_cleaned), 2)}%] {path} ===========")
    run_sphi_decomposition.main(
        img_path=path,
        layer_label="proj_14.95_to_15.05_um",
        low_cutoff_phi=low_cutoff_phi_all,
        high_cutoff_phi=high_cutoff_phi_all,
        profile_bins=20, q_decomp_radius=q_decomp_radius_all, show_figures=True
    )

In [ ]:

for i, path in enumerate(["/Users/andreadi/Desktop/Gas2.tif"]):
    run_tan_orient_extract.main(img_path=path, layer_label=layer_label_all, render=False, patch_mode="radius",
                                patch_size=dir_extr_patch_size_all, normal_validity_k=20, normal_validity_thresh=0.99,
                                compute_num=3000, grid_n_2dcurve_analysis=30, debug_2dcurve_analysis=False,
                                show_figures=False)
    run_sphi_decomposition.main(
        img_path=path,
        layer_label="proj_14.95_to_15.05_um",
        low_cutoff_phi=-np.pi / 4,
        high_cutoff_phi=np.pi / 4,
        profile_bins=20, q_decomp_radius=q_decomp_radius_all, show_figures=True
    )

## Create Batch Datasets

In [ ]:

all_morpho_profiles = []
for i, path in enumerate(img_path_list_exp4_cleaned):
    print(f"=========== [Progress {100 * np.round(i / len(img_path_list_exp4_cleaned), 2)}%] {path} ===========")
    resdata_dir, resfig_dir = datahandler.create_resdirs(path)
    resdata_dir_layer = os.path.join(resdata_dir, layer_label_all)
    path_parts = path.split(os.sep)
    gas_id = str(path_parts[-1].replace(".tif", ""))
    size_val = int(path_parts[-2])
    t_val = int(path_parts[-3])

    print(t_val, size_val, gas_id)
    mesh_s_rho_phi = datahandler.load_array(name="mesh_s-rho-phi", folderpath=resdata_dir)
    mesh = datahandler.load_mesh(filepath=os.path.join(resdata_dir, "sampling_mesh.ply"), recalc_normals=False,
                                 clean=False)
    s_vals = mesh_s_rho_phi[:, 0]
    rho_vals = mesh_s_rho_phi[:, 1]
    phi_vals = mesh_s_rho_phi[:, 2]

    s_max = s_vals.max()
    s_norm = s_vals / s_max if s_max > 0 else s_vals

    # Append every point to the list
    for j in range(len(s_vals)):
        all_morpho_profiles.append({
            "path": path,
            "Time": t_val,
            "Size": size_val,
            "GasID": gas_id,
            "s": s_vals[j],
            "s_normalised": s_norm[j],
            "rho": rho_vals[j],
            "phi_angle": phi_vals[j],
            "Volume": mesh.volume,
            "Area": mesh.area
        })
df_morpho_full = pd.DataFrame(all_morpho_profiles)
df_morpho_full.to_pickle(os.path.join(gastruloid_batch_output_folder, dataset_morphology_name))
print(f"Extraction Complete!")

In [ ]:

all_binned_profiles = []
all_raw_pointwise_nematic = []
for i, path in enumerate(img_path_list_exp4_cleaned):
    print(f"=========== [Progress {100 * np.round(i / len(img_path_list_exp4_cleaned), 2)}%] {path} ===========")
    resdata_dir, resfig_dir = datahandler.create_resdirs(path)
    resdata_dir_layer = os.path.join(resdata_dir, layer_label_all)
    path_parts = path.split(os.sep)
    gas_id = str(path_parts[-1].replace(".tif", ""))
    size_val = int(path_parts[-2])
    t_val = int(path_parts[-3])
    print(t_val, size_val, gas_id)
    coords = datahandler.load_array("dir_s-rho-phi_cropped", folderpath=resdata_dir_layer)
    q_data = np.load(os.path.join(resdata_dir_layer, f"q_sphi_full_cropped_{q_decomp_label_all}.npz"))
    q_tensor = q_data['q_sphi_cropped']
    s_bins = datahandler.load_array(f"s_bin_centers_cropped_{q_decomp_label_all}", folderpath=resdata_dir_layer)
    q_ss_m = datahandler.load_array(f"q_ss_mean_cropped_{q_decomp_label_all}", folderpath=resdata_dir_layer)
    q_pp_m = datahandler.load_array(f"q_phiphi_mean_cropped_{q_decomp_label_all}", folderpath=resdata_dir_layer)
    q_sp_m = datahandler.load_array(f"q_sphi_mean_cropped_{q_decomp_label_all}", folderpath=resdata_dir_layer)

    # --- Populate Binned Dataframe ---
    for j in range(len(s_bins)):
        all_binned_profiles.append({
            "path": path,
            "Time": t_val,
            "Size": size_val,
            "GasID": gas_id,
            "s_bin": s_bins[j],
            "Q_ss_mean": q_ss_m[j],
            "Q_phiphi_mean": q_pp_m[j],
            "Q_sphi_mean": q_sp_m[j],
        })

    # --- Populate Raw Dataframe ---
    # coords is [s, rho, phi]
    for k in range(len(coords)):
        all_raw_pointwise_nematic.append({
            "path": path,
            "Time": t_val,
            "Size": size_val,
            "GasID": gas_id,
            "dir_s": coords[k, 0],
            "dir_rho": coords[k, 1],
            "dir_phi": coords[k, 2],
            "Q_ss_raw": q_tensor[k, 0, 0],
            "Q_phiphi_raw": q_tensor[k, 1, 1],
            "Q_sphi_raw": q_tensor[k, 0, 1]
        })

# --- 5. Dataframe Creation ---
df_nematic_binned = pd.DataFrame(all_binned_profiles)
df_nematic_raw = pd.DataFrame(all_raw_pointwise_nematic)
df_nematic_binned.to_pickle(os.path.join(gastruloid_batch_output_folder, dataset_nematic_binned_name))
df_nematic_raw.to_pickle(os.path.join(gastruloid_batch_output_folder, dataset_nematic_profile_name))
print(f"Extraction Complete!")

## Batch Morphology Analysis

In [ ]:
# LOAD
df_morpho_full = pd.read_pickle(os.path.join(gastruloid_batch_output_folder, dataset_morphology_name))

In [ ]:
# --- 1. COLLAPSE TO UNIQUE GASTRULOIDS ---
# We keep only one entry per File to get the true biological N
df_unique = df_morpho_full.drop_duplicates(subset=["path"])

# --- 2. PLOTTING ---
unique_sizes = sorted(df_unique["Size"].unique())
colors = sns.color_palette("Set1", n_colors=len(unique_sizes))

fig, ax = plt.subplots(figsize=(6, 5))

sns.countplot(
    data=df_unique,  # Use the collapsed dataframe here
    x="Time",
    hue="Size",
    palette="Set1",
    hue_order=unique_sizes,
    edgecolor="black",
    linewidth=0.8,
    alpha=0.85,
    ax=ax
)

# --- 3. ANNOTATIONS (True N) ---
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f'{int(height)}',
                    (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='center',
                    xytext=(0, 8),
                    textcoords='offset points',
                    fontsize=10,
                    weight='bold')

# --- 4. STYLING ---
ax.set_title(r"$\text{Experimental Sample Size Overview}$", pad=20, fontsize=14)
ax.set_ylabel(r"$\text{Number of Gastruloids } (N)$", fontsize=12)
ax.set_xlabel(r"$\text{Time point } (t \text{ [h]})$", fontsize=12)
ax.legend(title=r"$\text{Seeding Size}$", frameon=False, loc='upper left')

sns.despine()
plt.tight_layout()

# Save using the specific path
plt.savefig(os.path.join(gastruloid_batch_output_folder, "sample_size_statistics.png"), bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# --- 1. PREPARE FILTERED DATA ---
phi_cutoff = np.pi / 3
df_top = df_morpho_full[df_morpho_full['phi_angle'].abs() <= phi_cutoff].copy()

# Setup common visual mappings for the profile plots
unique_times = np.sort(df_morpho_full['Time'].unique())
unique_sizes = np.sort(df_morpho_full['Size'].unique())
colors = plt.get_cmap('coolwarm', len(unique_times))(np.linspace(0, 1, len(unique_times)))
time_to_color = {t: colors[i] for i, t in enumerate(unique_times)}
size_to_style = {sz: style for sz, style in zip(unique_sizes, ['-', '--', ':', '-.'])}


# --- 2. AGGREGATE SUMMARY METRICS ---
def get_windowed_mid_thickness(group_top, group_full, window):
    mid_s = group_full['s'].max() / 2
    mask = (group_top['s'] >= (mid_s - window)) & (group_top['s'] <= (mid_s + window))
    if not mask.any():
        return group_top.loc[(group_top['s'] - mid_s).abs().idxmin(), 'rho']
    return group_top.loc[mask, 'rho'].mean()


summary_rows = []
mid_body_window_microns = 30

for path, group_full in df_morpho_full.groupby("path"):
    group_top = df_top[df_top["path"] == path]
    if group_top.empty: continue

    summary_rows.append({
        "path": path,
        "Time": group_full["Time"].iloc[0],
        "Size": group_full["Size"].iloc[0],
        "Area": group_full["Area"].iloc[0],
        "Volume": group_full["Volume"].iloc[0],
        "Body Length": group_full["s"].max(),  # From FULL mesh
        "Max Body Thickness": group_top["rho"].max(),  # From TOP half
        "Average Body Thickness": group_top["rho"].mean(),  # From TOP half
        "Mid Body Thickness": get_windowed_mid_thickness(group_top, group_full, mid_body_window_microns)
    })

df_summary = pd.DataFrame(summary_rows)

# Normalise Summary DF
metrics = ["Area", "Volume", "Body Length", "Max Body Thickness", "Average Body Thickness", "Mid Body Thickness"]
df_summary_norm = df_summary.copy()

for size in df_summary_norm["Size"].unique():
    size_mask = df_summary_norm["Size"] == size
    t_min = df_summary_norm.loc[size_mask, "Time"].min()
    baseline = df_summary_norm.loc[size_mask & (df_summary_norm["Time"] == t_min), metrics].mean()
    df_summary_norm.loc[size_mask, metrics] = df_summary_norm.loc[size_mask, metrics] / baseline


# --- 3. PLOTTING FUNCTIONS ---

def plot_advanced_morphometrics(df_plot, save_path, normalise=False):
    ylabels = [fr"Rel. {m} (norm. to $t_{{min}}$)" for m in metrics] if normalise else [
        r"Area ($\mu m^2$)", r"Volume ($\mu m^3$)", r"Body Length ($\mu m$)",
        r"Max Body Thickness ($\mu m$)", r"Avg. Body Thickness ($\mu m$)",
        fr"Mid Body Thickness ($\pm${mid_body_window_microns}$\mu m$)"
    ]
    suffix = "_normalised" if normalise else ""
    title_prefix = "Normalised" if normalise else "Gastruloid"
    custom_palette = sns.color_palette(["#E41A1C", "#377EB8"])

    for metric, ylabel in zip(metrics, ylabels):
        fig, ax = plt.subplots(figsize=(6, 5))
        sns.boxplot(data=df_plot, x="Time", y=metric, hue="Size", palette=custom_palette, ax=ax, fliersize=0,
                    boxprops={'alpha': 0.3}, zorder=1)
        sns.stripplot(data=df_plot, x="Time", y=metric, hue="Size", palette=custom_palette, ax=ax, dodge=True,
                      alpha=0.7, size=6, edgecolor='white', linewidth=0.5, zorder=2)
        # Fixed pointplot warnings
        sns.pointplot(data=df_plot, x="Time", y=metric, hue="Size", palette=custom_palette, ax=ax, dodge=0.4,
                      estimator=np.mean, errorbar=None, markers="o", markersize=7, linewidth=2, zorder=3)

        ax.set_title(fr"$\text{{{title_prefix} {metric}}}$", pad=12, fontsize=14)
        ax.set_ylabel(ylabel, fontsize=12)
        ax.set_xlabel(r"Time point ($t$ [h])", fontsize=12)

        if normalise: ax.axhline(1, color='black', linestyle='--', alpha=0.5, linewidth=1.5, zorder=0)

        handles, labels = ax.get_legend_handles_labels()
        ax.legend(handles[:len(unique_sizes)], labels[:len(unique_sizes)], title=r"$\text{Seeding Size}$",
                  frameon=False, loc='best')
        sns.despine()
        plt.tight_layout()
        plt.savefig(os.path.join(save_path, f"{metric.replace(' ', '_')}{suffix}.png"), bbox_inches='tight', dpi=300)
        plt.show()

# def plot_regression_grid(df_plot, save_path, normalise=False):
#     ylabels = [fr"Rel. {m} (norm. to $t_{{min}}$)" for m in metrics] if normalise else [
#         r"Area ($\mu m^2$)", r"Volume ($\mu m^3$)", r"Body Length ($\mu m$)",
#         r"Max Body Thickness ($\mu m$)", r"Avg. Body Thickness ($\mu m$)",
#         fr"Mid Body Thickness ($\pm${mid_body_window_microns}$\mu m$)"
#     ]
#     suffix = "_normalised" if normalise else ""
#     title_prefix = "Normalised" if normalise else "Gastruloid"
#
#     fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharex=True)
#     axes_flat = axes.ravel()
#     colors_reg = sns.color_palette("Set1", n_colors=len(unique_sizes))
#
#     for i, metric in enumerate(metrics):
#         ax = axes_flat[i]
#         for size, color in zip(unique_sizes, colors_reg):
#             subset = df_plot[df_plot["Size"] == size]
#             sns.regplot(data=subset, x="Time", y=metric, ax=ax, color=color,
#                         scatter_kws={"s": 50, "edgecolor": "w", "alpha": 0.7, "linewidths": 0.5},
#                         line_kws={"linewidth": 2}, label=f"Size {size}", x_jitter=1.5)
#
#         if normalise: ax.axhline(1, color='black', linestyle='--', alpha=0.3, linewidth=1)
#         ax.set_title(fr"$\text{{{title_prefix} {metric}}}$", pad=12, fontsize=14)
#         ax.set_ylabel(ylabels[i], fontsize=11)
#         if i >= 3: ax.set_xlabel(r"Time point ($t$ [h])", fontsize=12)
#
#     handles, labels = axes_flat[0].get_legend_handles_labels()
#     by_label = dict(zip(labels, handles))
#     fig.legend(by_label.values(), by_label.keys(), title=r"$\text{Seeding Size}$", loc='center right',
#                bbox_to_anchor=(1.0, 0.5), frameon=False)
#     sns.despine()
#     plt.tight_layout()
#     plt.subplots_adjust(right=0.92, bottom=0.12, hspace=0.25)
#     plt.savefig(os.path.join(save_path, f"morphometrics_regression{suffix}.png"), bbox_inches='tight', dpi=300)
#     plt.show()
# print(">> Plotting Regression Grids...")
# plot_regression_grid(df_summary, gastruloid_batch_output_folder, normalise=False)
# plot_regression_grid(df_summary_norm, gastruloid_batch_output_folder, normalise=True)


In [ ]:
# --- 4. EXECUTION BLOCK ---
print(">> Plotting Advanced Morphometrics...")
plot_advanced_morphometrics(df_summary, gastruloid_batch_output_folder, normalise=False)
plot_advanced_morphometrics(df_summary_norm, gastruloid_batch_output_folder, normalise=True)

In [ ]:
def add_custom_legend(fig, ax):
    size_handles = [Line2D([0], [0], color='gray', linestyle=size_to_style[sz], label=f"Size {sz}") for sz in
                    unique_sizes]
    fig.legend(handles=size_handles, loc='upper right', bbox_to_anchor=(0.98, 0.85), title=r"$\text{Seeding Size}$",
               frameon=False)
    sm = cm.ScalarMappable(cmap=ListedColormap(colors),
                           norm=BoundaryNorm(np.arange(len(unique_times) + 1) - 0.5, len(unique_times)))
    cbar_ax = fig.add_axes([0.88, 0.15, 0.02, 0.5])
    cbar = fig.colorbar(sm, cax=cbar_ax, ticks=np.arange(len(unique_times)))
    cbar.ax.set_yticklabels([str(int(t)) for t in unique_times])
    cbar.set_label(r"$\text{Time (h)}$")
    sns.despine(ax=ax)


def plot_individual_thickness_profiles(df, save_path, use_normalised=True):
    fig, ax = plt.subplots(figsize=(10, 6))
    plt.subplots_adjust(right=0.85)
    x_col = "s_normalised" if use_normalised else "s"

    for file_id, data in df.groupby("path"):
        t_val = data["Time"].iloc[0]
        sz_val = data["Size"].iloc[0]
        stats = data.groupby(x_col)["rho"].agg(["mean", "std"]).reset_index()
        color = time_to_color[t_val]
        style = size_to_style[sz_val]
        ax.plot(stats[x_col], stats["mean"], color=color, linestyle=style, linewidth=1, alpha=0.6)

    if use_normalised:
        ax.set_xlabel(r"Normalised Arc Length ($s/s_{max}$)")
        suffix = "normalised"
    else:
        ax.set_xlabel(r"Arc Length $s$ ($\mu m$)")
        suffix = "physical"

    ax.set_ylabel(r"Radius $\rho$ ($\mu m$)")
    ax.set_title(fr"$\text{{Individual Thickness Profiles ({suffix}, top half)}}$")
    add_custom_legend(fig, ax)
    plt.savefig(os.path.join(save_path, f"individual_rho-profiles_{suffix}.png"), bbox_inches='tight', dpi=200)
    plt.show()


print(">> Plotting Individual Variance Profiles...")
plot_individual_thickness_profiles(df_top, gastruloid_batch_output_folder, use_normalised=True)
plot_individual_thickness_profiles(df_top, gastruloid_batch_output_folder, use_normalised=False)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.interpolate import UnivariateSpline
from matplotlib.lines import Line2D


def add_custom_legend(fig, ax, unique_sizes, unique_times, size_to_style, time_to_color):
    """Adds the seeding size legend and the time colorbar."""
    # 1. Size Legend
    size_handles = [Line2D([0], [0], color='gray', linestyle=size_to_style[sz],
                           label=f"Size {sz}") for sz in unique_sizes]
    fig.legend(handles=size_handles, loc='upper right', bbox_to_anchor=(0.98, 0.85),
               title=r"$\text{Seeding Size}$", frameon=False)

    # 2. Time Colorbar
    from matplotlib import cm
    from matplotlib.colors import ListedColormap, BoundaryNorm

    # Extract colors from your time_to_color map
    colors = [time_to_color[t] for t in unique_times]
    sm = cm.ScalarMappable(cmap=ListedColormap(colors),
                           norm=BoundaryNorm(np.arange(len(unique_times) + 1) - 0.5, len(unique_times)))

    cbar_ax = fig.add_axes([0.88, 0.15, 0.02, 0.5])
    cbar = fig.colorbar(sm, cax=cbar_ax, ticks=np.arange(len(unique_times)))
    cbar.ax.set_yticklabels([str(int(t)) for t in unique_times])
    cbar.set_label(r"$\text{Time (h)}$")


def plot_fitted_thickness_profiles(df, save_path):
    fig, ax = plt.subplots(figsize=(10, 6))
    plt.subplots_adjust(right=0.85)

    unique_times = sorted(df["Time"].unique())
    unique_sizes = sorted(df["Size"].unique())

    for (t, sz), group in df.groupby(["Time", "Size"]):

        # --- 1. CALCULATE NORMALISED S PER GASTULOID ---
        # We divide each s by the max s of its specific path
        group = group.copy()
        group["s_norm"] = group.groupby("path")["s"].transform(lambda x: x / x.max())

        # --- 2. BINNED AVERAGING ON NORMALISED SCALE ---
        num_bins = 200
        # Normalised scale is always 0 to 1
        bins = np.linspace(0, 1.0, num_bins)
        bin_centers = (bins[:-1] + bins[1:]) / 2
        bin_idx = np.digitize(group["s_norm"].values, bins)

        binned_mu = []
        for i in range(1, len(bins)):
            vals = group["rho"].values[bin_idx == i]
            if len(vals) > 5:
                binned_mu.append(np.mean(vals))
            else:
                binned_mu.append(np.nan)

        binned_mu = np.array(binned_mu)
        valid = ~np.isnan(binned_mu)
        if not np.any(valid): continue

        plot_s = bin_centers[valid]
        plot_mu = binned_mu[valid]

        # --- 3. FIT SMOOTH SPLINE ---
        # s=num_bins*0.1 is a good starting point for normalised data
        spline = UnivariateSpline(plot_s, plot_mu, k=3, s=num_bins * 0.1)

        # High resolution sampling for the plot
        s_smooth = np.linspace(0, 1.0, 1000)
        mu_smooth = spline(s_smooth)

        color = time_to_color[t]
        style = size_to_style[sz]

        ax.plot(s_smooth, mu_smooth, color=color, linestyle=style,
                linewidth=3, alpha=0.9)

    ax.set_xlabel(r"Normalised Arc Length ($s/s_{max}$)")
    ax.set_ylabel(r"Radius $\rho$ ($\mu m$)")
    ax.set_title(r"$\text{Averaged Normalised Morphological Profiles}$")
    ax.set_xlim(0, 1.0)
    ax.set_ylim(bottom=0)

    add_custom_legend(fig, ax, unique_sizes, unique_times, size_to_style, time_to_color)
    sns.despine(offset=5)

    plt.savefig(os.path.join(save_path, "representative_profiles_normalised.png"), bbox_inches='tight', dpi=300)
    plt.show()


# Run the complete code
plot_fitted_thickness_profiles(df_top, gastruloid_batch_output_folder)

In [ ]:
def find_hinge_point(group, window=(0.3, 0.7)):
    if group["Time"].iloc[0] <= 72: return 0.5
    stats = group.groupby("s_normalised")["rho"].mean().reset_index()
    s_norm = stats["s_normalised"].values
    rho = gaussian_filter1d(stats["rho"].values, sigma=5)
    mask = (s_norm >= window[0]) & (s_norm <= window[1])
    if not mask.any(): return 0.5
    return s_norm[mask][np.argmin(rho[mask])]


hinge_map = df_top.groupby("path").apply(find_hinge_point, include_groups=False).to_dict()
df_top["s_hinge"] = df_top["path"].map(hinge_map)


def plot_hinge_aligned_profiles(df, save_path):
    fig, ax = plt.subplots(figsize=(12, 6))
    plt.subplots_adjust(right=0.85)

    for file_id, data in df.groupby("path"):
        data = data.copy()
        data["s_aligned"] = data["s_normalised"] - data["s_hinge"]
        stats = data.groupby("s_aligned")["rho"].agg(["mean", "std"]).reset_index()
        mu = gaussian_filter1d(stats["mean"], sigma=3)
        t_val = data["Time"].iloc[0]
        sz_val = data["Size"].iloc[0]
        ax.plot(stats["s_aligned"], mu, color=time_to_color[t_val], linestyle=size_to_style[sz_val], alpha=0.6)

    ax.axvline(0, color='black', linestyle=':', alpha=0.5, label="Hinge Alignment")
    ax.set_xlabel(r"Aligned Arc Length ($s_{norm} - s_{hinge}$)")
    ax.set_ylabel(r"Radius $\rho$ ($\mu m$)")
    ax.set_title(r"$\text{Profiles Aligned by Mid-body Hinge (top half)}$")
    add_custom_legend(fig, ax)
    plt.savefig(os.path.join(save_path, "hinge_aligned_profiles.png"), bbox_inches='tight', dpi=200)
    plt.show()


def plot_mean_hinge_aligned_physical(df, save_path):
    fig, ax = plt.subplots(figsize=(10, 6))
    plt.subplots_adjust(right=0.85)
    s_align_grid = np.linspace(-1.0, 1.0, 500)

    for t in unique_times:
        for sz in unique_sizes:
            group_data = df[(df["Time"] == t) & (df["Size"] == sz)]
            if group_data.empty: continue
            interp_profiles = []

            for file_id, gast_data in group_data.groupby("path"):
                gast_data = gast_data.copy()
                gast_data["s_aligned"] = gast_data["s_normalised"] - gast_data["s_hinge"].iloc[0]
                stats = gast_data.groupby("s_aligned")["rho"].mean().reset_index().sort_values("s_aligned")
                f = interp1d(stats["s_aligned"], stats["rho"], bounds_error=False, fill_value=np.nan)
                interp_profiles.append(f(s_align_grid))

            with np.errstate(all='ignore'):
                mean_shape = np.nanmean(interp_profiles, axis=0)

            mask = ~np.isnan(mean_shape)
            if not np.any(mask): continue
            y_smooth = gaussian_filter1d(mean_shape[mask], sigma=3)
            ax.plot(s_align_grid[mask], y_smooth, color=time_to_color[t], linestyle=size_to_style[sz], linewidth=3,
                    alpha=0.9)

    ax.axvline(0, color='black', linestyle=':', alpha=0.3)
    ax.set_xlabel(r"Aligned Normalised Arc Length ($s_{norm} - s_{hinge}$)")
    ax.set_ylabel(r"Radius $\rho$ ($\mu m$)")
    ax.set_title(r"$\text{Mean Hinge-Aligned Morphological Profiles (top half)}$")
    add_custom_legend(fig, ax)
    plt.savefig(os.path.join(save_path, "hinge_aligned_mean_physical.png"), bbox_inches='tight', dpi=300)
    plt.show()


print(">> Plotting Hinge-Aligned Profiles...")
plot_hinge_aligned_profiles(df_top, gastruloid_batch_output_folder)
plot_mean_hinge_aligned_physical(df_top, gastruloid_batch_output_folder)

## Batch Nematic Analysis

In [ ]:
df_nematic_binned = pd.read_pickle(os.path.join(gastruloid_batch_output_folder, dataset_nematic_binned_name))
df_nematic_raw = pd.read_pickle(os.path.join(gastruloid_batch_output_folder, dataset_nematic_profile_name))

In [ ]:
unique_times = np.sort(df_nematic_raw['Time'].unique())
unique_sizes = np.sort(df_nematic_raw['Size'].unique())
colors = plt.get_cmap('coolwarm', len(unique_times))(np.linspace(0, 1, len(unique_times)))
time_to_color = {t: colors[i] for i, t in enumerate(unique_times)}
size_to_style = {sz: style for sz, style in zip(unique_sizes, ['-', '--', ':', '-.'])}


def plot_clean_raw_nematic_analysis(df_raw, save_path, use_normalised=True):
    """
    Creates ultra-clean analysis plots from the raw point-wise nematic data.
    """
    df = df_raw.copy()

    # 1. Calculate Magnitude S from raw components
    df['S_magnitude'] = 2 * np.sqrt(df['Q_ss_raw'] ** 2 + df['Q_sphi_raw'] ** 2)

    # 2. X-axis Toggle using 'dir_s'
    if use_normalised:
        # Calculate s_norm per file based on the max dir_s found
        df['s_plot'] = df.groupby("path")['dir_s'].transform(lambda x: x / x.max())
        x_label = r"Normalised Arc Length ($s/s_{max}$)"
        suffix = "norm"
    else:
        df['s_plot'] = df['dir_s']
        x_label = r"Arc Length $s$ ($\mu m$)"
        suffix = "phys"

    # --- Setup Figure ---
    fig, axes = plt.subplots(4, 1, figsize=(8, 14), sharex=True)
    plt.subplots_adjust(right=0.82, hspace=0.15)

    components = [
        ('S_magnitude', r'Order Magnitude $S$', (0, 1.05)),
        ('Q_phiphi_raw', r'Circumferential $Q_{\phi\phi}$', (-0.55, 0.55)),
        ('Q_ss_raw', r'Longitudinal $Q_{ss}$', (-0.55, 0.55)),
        ('Q_sphi_raw', r'Shear $Q_{s\phi}$', (-0.55, 0.55))
    ]

    # --- Plotting Loop ---
    for i, (col, label, ylims) in enumerate(components):
        ax = axes[i]

        # Group by experimental conditions
        for (t_val, sz_val), group in df.groupby(['Time', 'Size']):
            # Sort raw points by position
            group = group.dropna(subset=['s_plot', col]).sort_values('s_plot')

            # To remove the "jagged" look, we resample the raw points
            # into a high-density linear space (e.g., 200 points)
            x_eval = np.linspace(group['s_plot'].min(), group['s_plot'].max(), 200)

            # Interpolate raw values onto the dense grid
            y_interp = np.interp(x_eval, group['s_plot'], group[col])

            # Apply a significant Gaussian filter to get a smooth "trend" line
            # Increase sigma (e.g., to 10 or 12) for even smoother lines
            y_smooth = gaussian_filter1d(y_interp, sigma=10)

            ax.plot(x_eval, y_smooth,
                    color=time_to_color[t_val],
                    linestyle=size_to_style[sz_val],
                    linewidth=3, alpha=0.9)

        ax.set_ylabel(label)
        ax.set_ylim(ylims)
        if col != 'S_magnitude':
            ax.axhline(0, color='black', linewidth=1, alpha=0.2)
        sns.despine(ax=ax)

    axes[-1].set_xlabel(x_label)

    # --- Legend & Colorbar (Re-using your setup) ---
    size_handles = [Line2D([0], [0], color='gray', linestyle=size_to_style[sz], label=f"Size {sz}") for sz in
                    unique_sizes]
    fig.legend(handles=size_handles, loc='upper right', bbox_to_anchor=(0.98, 0.8), title=r"$\text{Seeding Size}$",
               frameon=False)

    sm = cm.ScalarMappable(cmap=ListedColormap(colors),
                           norm=BoundaryNorm(np.arange(len(unique_times) + 1) - 0.5, len(unique_times)))
    cbar_ax = fig.add_axes([0.88, 0.35, 0.02, 0.3])
    cbar = fig.colorbar(sm, cax=cbar_ax, ticks=np.arange(len(unique_times)))
    cbar.ax.set_yticklabels([str(int(t)) for t in unique_times])
    cbar.set_label(r"$\text{Time (h)}$")

    # Save
    plt.savefig(os.path.join(save_path, f"nematic-profiles_{layer_label_all}_r={q_decomp_radius_all}_{suffix}.png"),
                bbox_inches='tight')
    plt.show()


# --- RUN ---
plot_clean_raw_nematic_analysis(df_nematic_raw, gastruloid_batch_output_folder, use_normalised=True)
plot_clean_raw_nematic_analysis(df_nematic_raw, gastruloid_batch_output_folder, use_normalised=False)

In [ ]:
def plot_nematic_phase_diagram(df_raw, save_path, component='Q_phiphi_raw', ylims=(-0.55, 0.55), ydashed=0.0):
    """
    Creates a grid of plots (Time vs Size) showing individual gastruloid profiles
    for a specific nematic component.
    """
    df = df_raw.copy()

    # 1. Ensure Normalised Coordinate
    df['s_plot'] = df.groupby("path")['dir_s'].transform(lambda x: x / x.max())

    unique_times = np.sort(df['Time'].unique())
    unique_sizes = np.sort(df['Size'].unique())

    # Create the figure grid
    n_rows = len(unique_sizes)
    n_cols = len(unique_times)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3.5 * n_rows),
                             sharex=True, sharey=True)

    # Handle single row/col cases so indexing doesn't break
    if n_rows == 1: axes = np.expand_dims(axes, axis=0)
    if n_cols == 1: axes = np.expand_dims(axes, axis=1)

    # --- Plotting Loop ---
    for r, sz_val in enumerate(unique_sizes):
        for c, t_val in enumerate(unique_times):
            ax = axes[r, c]

            # Filter data for this specific cell in the phase diagram
            cell_data = df[(df['Time'] == t_val) & (df['Size'] == sz_val)]

            if cell_data.empty:
                ax.axis('off')
                continue

            # Plot individual gastruloids as thin lines to show variance
            for file_id, gast_data in cell_data.groupby("path"):
                gast_data = gast_data.dropna(subset=['s_plot', component]).sort_values('s_plot')

                if len(gast_data) < 10: continue

                # Smooth individual profile
                x_eval = np.linspace(0, 1, 200)
                y_interp = np.interp(x_eval, gast_data['s_plot'], gast_data[component])
                y_smooth = gaussian_filter1d(y_interp, sigma=8)

                ax.plot(x_eval, y_smooth, color=time_to_color[t_val], alpha=0.4, linewidth=1.5)

            # Plot the population mean as a thick line
            # Grouping by a binned version of s_plot to get the mean trend
            cell_data['s_bin'] = pd.cut(cell_data['s_plot'], bins=np.linspace(0, 1, 50))
            mean_trend = cell_data.groupby('s_bin')[component].mean().values
            x_mean = np.linspace(0, 1, len(mean_trend))

            # Smooth the mean
            mean_smooth = gaussian_filter1d(mean_trend, sigma=2)
            ax.plot(x_mean, mean_smooth, color='black', linewidth=2.5, linestyle='-')

            # Aesthetics per subplot
            ax.set_ylim(ylims)
            ax.axhline(ydashed, color='black', linewidth=0.8, alpha=0.3)
            sns.despine(ax=ax)

            # Label only the edges
            if r == 0:
                ax.set_title(f"Time: {t_val}h", fontsize=14, pad=10)
            if c == n_cols - 1:
                ax.text(1.05, 0.5, f"Size: {sz_val}", transform=ax.transAxes,
                        rotation=270, va='center', fontsize=14, fontweight='bold')

    # Global Labels
    fig.supxlabel(r"Normalised Arc Length ($s/s_{max}$)", fontsize=16)
    fig.supylabel(f"{component}", fontsize=16)

    plt.tight_layout()
    plt.subplots_adjust(top=0.92, right=0.92)

    filename = f"phase_diagram_{component}.png"
    plt.savefig(os.path.join(save_path, filename), dpi=300)
    plt.show()


# --- RUN ---
# You can easily swap components here
plot_nematic_phase_diagram(df_nematic_raw, gastruloid_batch_output_folder, component='Q_phiphi_raw')
plot_nematic_phase_diagram(df_nematic_raw, gastruloid_batch_output_folder, component='Q_ss_raw')
plot_nematic_phase_diagram(df_nematic_raw, gastruloid_batch_output_folder, component='Q_sphi_raw')
df_nematic_raw['S'] = 2 * np.sqrt(df_nematic_raw['Q_ss_raw'] ** 2 + df_nematic_raw['Q_sphi_raw'] ** 2)
plot_nematic_phase_diagram(df_nematic_raw, gastruloid_batch_output_folder, component='S', ylims=[-0.01, 1.01],
                           ydashed=1.0)